# 07 — Static Arbitrage Diagnostics for Synthetic European Option Surfaces

## Research purpose

This notebook develops a controlled diagnostic framework for detecting **static no-arbitrage violations** in European option price and implied-volatility surfaces.

The goal is not to clean market data, fit SVI, calibrate Heston/Bates, or repair a broken surface. The goal is to build and validate the diagnostic layer first, using synthetic surfaces where the correct answer is known.

This notebook extends the previous implied-volatility notebook from **single-option admissibility** to **surface-level consistency**.

Notebook 06 asked:

> Is this individual option price admissible?

Notebook 07 asks:

> Is the entire option surface internally consistent across strikes and maturities?

## Research question

Given a grid of European option prices or implied volatilities across strikes and maturities, can we detect whether the surface violates static no-arbitrage structure?

The notebook will test this question using:

1. a clean Black-Scholes surface as a positive control;
2. deliberately corrupted price surfaces as negative controls;
3. deliberately corrupted implied-volatility / total-variance surfaces as negative controls.

A successful diagnostic framework must satisfy both conditions:

* it must **not falsely reject** a clean internally consistent surface;
* it must **correctly flag** deliberately constructed violations.

## Scope

This notebook studies European options under controlled assumptions:

* deterministic spot price;
* deterministic risk-free rate;
* deterministic dividend yield;
* European exercise only;
* synthetic strike and maturity grids;
* known model-generated prices;
* known deliberate violations.

No empirical option-chain data is used in this notebook.

## Diagnostics covered

The notebook will evaluate the following static-arbitrage diagnostics:

1. **Pointwise option bounds**

   * call lower and upper bounds;
   * put lower and upper bounds.

2. **Put-call parity**

   * consistency between calls, puts, forwards, rates, dividends, and discounting.

3. **Vertical-spread monotonicity**

   * call prices must be nonincreasing in strike for fixed maturity;
   * put prices must be nondecreasing in strike for fixed maturity.

4. **Butterfly convexity**

   * call and put prices must be convex in strike for fixed maturity;
   * convexity failures indicate local butterfly-arbitrage candidates and negative-density pathologies.

5. **Calendar / term-structure consistency**

   * controlled calendar checks across maturities;
   * total implied variance should be nondecreasing in maturity when compared at fixed log-moneyness.

6. **Implied-volatility consistency**

   * implied volatility must be finite, positive, and recoverable from admissible prices;
   * total variance must be nonnegative and maturity-consistent.

## Allowed claims

If the notebook passes, the following claims are allowed:

* A clean Black-Scholes-generated European option surface passes the implemented diagnostics up to numerical tolerance.
* Deliberately corrupted surfaces trigger the intended diagnostic failures.
* The diagnostic framework can identify pointwise bound violations, parity violations, vertical-spread violations, butterfly-convexity violations, and controlled calendar / total-variance inconsistencies.
* The framework is ready to be reused later as a diagnostic layer before any empirical option-chain work or surface fitting.

## Not allowed claims

This notebook does **not** claim that:

* the diagnostics repair bad surfaces;
* the diagnostics clean real market quotes;
* the diagnostics validate SPY option-chain data;
* the diagnostics prove an SVI surface is arbitrage-free;
* the diagnostics justify Heston or Bates calibration;
* the diagnostics detect every possible form of dynamic arbitrage;
* the diagnostics handle American exercise, discrete dividends, stale quotes, bid-ask microstructure, early exercise, or liquidity effects.

## Methodological rule

The notebook must pass both positive and negative controls.

A clean synthetic surface passing the diagnostics is necessary but not sufficient. The diagnostics must also catch deliberately constructed violations.

Final readiness requires:

* clean benchmark surface passes;
* bound violation is detected;
* put-call parity violation is detected;
* vertical-spread violation is detected;
* butterfly-convexity violation is detected;
* calendar / total-variance violation is detected;
* final diagnostic ledger reports all expected outcomes.

Only then can the notebook set:

`NOTEBOOK_07_READY = True`


In [1]:
# ============================================================
# 07 — Static Arbitrage Diagnostics for Synthetic Option Surfaces
# Cell 2: Imports, notebook configuration, and numerical tolerances
# ============================================================

from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from scipy.stats import norm


# ------------------------------------------------------------
# Notebook identity
# ------------------------------------------------------------

NOTEBOOK_NAME = "07_static_arbitrage_diagnostics_for_option_surfaces"
NOTEBOOK_VERSION = "v1.1"
NOTEBOOK_PURPOSE = (
    "Controlled static-arbitrage diagnostics for synthetic European option surfaces."
)


# ------------------------------------------------------------
# Display and numerical settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.10f}")

np.set_printoptions(
    precision=10,
    suppress=True,
    linewidth=140,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ------------------------------------------------------------
# Global deterministic configuration
# ------------------------------------------------------------

# This notebook should be deterministic. No randomness is needed
# for the core synthetic-surface diagnostics.
RANDOM_SEED = 62007
np.random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# Numerical tolerance discipline
# ------------------------------------------------------------

@dataclass(frozen=True)
class DiagnosticTolerances:
    """
    Numerical tolerances used throughout the notebook.

    The notebook separates tiny floating-point residuals from economically
    meaningful violations. Synthetic data should pass under strict tolerances.
    Future empirical notebooks may need looser tolerances, but this notebook
    should stay strict because the data-generating process is controlled.
    """

    price_abs: float = 1.0e-10
    parity_abs: float = 1.0e-9
    monotonicity_abs: float = 1.0e-10
    convexity_abs: float = 1.0e-10
    calendar_abs: float = 1.0e-10
    total_variance_abs: float = 1.0e-10
    iv_abs: float = 1.0e-10
    finite_diff_abs: float = 1.0e-8


TOL = DiagnosticTolerances()


# ------------------------------------------------------------
# Baseline synthetic market assumptions
# ------------------------------------------------------------

# These are not empirical inputs. They define the controlled synthetic economy.
S0 = 100.0
r = 0.03
q = 0.01

# Maturities are intentionally simple and strictly increasing.
MATURITIES = np.array([0.10, 0.25, 0.50, 1.00, 2.00], dtype=float)

# Use log-moneyness as the primary surface coordinate.
# Strike grids will be generated from K = F(T) * exp(k_log).
K_LOG_GRID = np.array(
    [-0.40, -0.30, -0.20, -0.10, -0.05, 0.00, 0.05, 0.10, 0.20, 0.30, 0.40],
    dtype=float,
)

# Constant-volatility Black-Scholes surface for the clean positive control.
BASE_VOL = 0.20


# ------------------------------------------------------------
# Diagnostic status labels
# ------------------------------------------------------------

STATUS_PASS = "PASS"
STATUS_FAIL = "FAIL"

SEVERITY_INFO = "INFO"
SEVERITY_LOW = "LOW"
SEVERITY_MEDIUM = "MEDIUM"
SEVERITY_HIGH = "HIGH"
SEVERITY_CRITICAL = "CRITICAL"


# ------------------------------------------------------------
# Startup audit
# ------------------------------------------------------------

startup_summary = pd.DataFrame(
    {
        "field": [
            "notebook",
            "version",
            "purpose",
            "spot",
            "risk_free_rate",
            "dividend_yield",
            "base_vol",
            "maturity_count",
            "log_moneyness_count",
            "random_seed",
        ],
        "value": [
            NOTEBOOK_NAME,
            NOTEBOOK_VERSION,
            NOTEBOOK_PURPOSE,
            S0,
            r,
            q,
            BASE_VOL,
            len(MATURITIES),
            len(K_LOG_GRID),
            RANDOM_SEED,
        ],
    }
)

startup_summary

,field,value
0,notebook,07_static_arbitrage_diagnostics_for_option_sur...
1,version,v1.1
2,purpose,Controlled static-arbitrage diagnostics for sy...
3,spot,100.0000000000
4,risk_free_rate,0.0300000000
5,dividend_yield,0.0100000000
6,base_vol,0.2000000000
7,maturity_count,5
8,log_moneyness_count,11
9,random_seed,62007


## 1. Mathematical setup

This notebook works in a controlled European option setting with deterministic rates and deterministic proportional dividends.

Let:

- $S_0$ be the current spot price;
- $r$ be the continuously compounded risk-free rate;
- $q$ be the continuously compounded dividend yield;
- $T$ be time to maturity;
- $K$ be the strike;
- $D(T)$ be the risk-free discount factor;
- $F(T)$ be the maturity-specific forward price;
- $k$ be log-moneyness relative to the forward.

The risk-free discount factor is:

$$
D(T) = e^{-rT}
$$

The maturity-specific forward price is:

$$
F(T) = S_0 e^{(r-q)T}
$$

The prepaid forward value of the stock is:

$$
S_0 e^{-qT}
$$

Using the discount factor and the forward price, the same prepaid forward value can be written as:

$$
D(T)F(T) = e^{-rT} S_0 e^{(r-q)T} = S_0 e^{-qT}
$$

Log-moneyness is defined by:

$$
k = \log\left(\frac{K}{F(T)}\right)
$$

Therefore, a strike can be generated from a maturity and a log-moneyness value by:

$$
K = F(T)e^k
$$

This notebook uses log-moneyness as the primary surface coordinate because maturity comparisons are cleaner when strikes are measured relative to the forward.

A fixed raw strike $K$ can represent different moneyness levels across maturities because $F(T)$ changes with $T$. By contrast, fixed $k$ means the option is compared at the same forward-relative location across the term structure.

---

## Static no-arbitrage objects

The primary diagnostic object is the European call surface:

$$
C(K,T)
$$

The European put surface is:

$$
P(K,T)
$$

For deterministic rates and proportional dividends, European put-call parity is:

$$
C(K,T) - P(K,T) = S_0 e^{-qT} - K e^{-rT}
$$

Using the discount factor and the forward price, this is equivalently:

$$
C(K,T) - P(K,T) = D(T)\left(F(T)-K\right)
$$

This parity relation links calls, puts, forwards, discounting, and dividends. In this synthetic notebook, parity should hold up to numerical tolerance unless the surface is deliberately corrupted.

---

## Pointwise call bounds

For each strike and maturity, a European call must satisfy:

$$
\max\left(S_0 e^{-qT} - K e^{-rT}, 0\right)
\leq
C(K,T)
\leq
S_0 e^{-qT}
$$

Using $D(T)$ and $F(T)$, the same bounds are:

$$
D(T)\max\left(F(T)-K,0\right)
\leq
C(K,T)
\leq
D(T)F(T)
$$

The lower bound is the discounted intrinsic value under the forward measure. The upper bound is the prepaid forward value of the stock.

---

## Pointwise put bounds

For each strike and maturity, a European put must satisfy:

$$
\max\left(K e^{-rT} - S_0 e^{-qT}, 0\right)
\leq
P(K,T)
\leq
K e^{-rT}
$$

Using $D(T)$ and $F(T)$, the same bounds are:

$$
D(T)\max\left(K-F(T),0\right)
\leq
P(K,T)
\leq
D(T)K
$$

The lower bound is the discounted intrinsic value under the forward measure. The upper bound is the discounted strike.

---

## Vertical-spread monotonicity

For fixed maturity $T$, European call prices must be nonincreasing in strike.

If:

$$
K_1 < K_2
$$

then:

$$
C(K_1,T) \geq C(K_2,T)
$$

Equivalently, adjacent call price differences across an increasing strike grid must satisfy:

$$
C(K_{i+1},T) - C(K_i,T) \leq 0
$$

For fixed maturity $T$, European put prices must be nondecreasing in strike.

If:

$$
K_1 < K_2
$$

then:

$$
P(K_1,T) \leq P(K_2,T)
$$

Equivalently, adjacent put price differences across an increasing strike grid must satisfy:

$$
P(K_{i+1},T) - P(K_i,T) \geq 0
$$

These are vertical-spread no-arbitrage conditions.

---

## Butterfly convexity

For fixed maturity $T$, European call and put prices must be convex functions of strike.

For a smooth call price curve, convexity requires:

$$
\frac{\partial^2 C}{\partial K^2}(K,T) \geq 0
$$

For a smooth put price curve, convexity requires:

$$
\frac{\partial^2 P}{\partial K^2}(K,T) \geq 0
$$

On a discrete strike grid, this notebook checks convexity using adjacent strike slopes rather than simple second differences. This avoids assuming that strikes are equally spaced.

For calls, define the adjacent strike slope:

$$
s_i^C(T)
=
\frac{C(K_{i+1},T)-C(K_i,T)}{K_{i+1}-K_i}
$$

Convexity requires these slopes to be nondecreasing as strike increases:

$$
s_{i+1}^C(T) - s_i^C(T) \geq 0
$$

For puts, define the adjacent strike slope:

$$
s_i^P(T)
=
\frac{P(K_{i+1},T)-P(K_i,T)}{K_{i+1}-K_i}
$$

Convexity requires:

$$
s_{i+1}^P(T) - s_i^P(T) \geq 0
$$

A convexity violation indicates a butterfly-arbitrage candidate. In risk-neutral density terms, call convexity is linked to nonnegative state-price density.

---

## Calendar and total-variance consistency

For implied-volatility surfaces, define total implied variance as:

$$
w(k,T) = \sigma_{\mathrm{imp}}^2(k,T)T
$$

For fixed log-moneyness $k$, absence of calendar-spread arbitrage requires total implied variance to be nondecreasing in maturity:

$$
T_1 < T_2
\quad \Rightarrow \quad
w(k,T_1) \leq w(k,T_2)
$$

Equivalently:

$$
w(k,T_2) - w(k,T_1) \geq 0
$$

This notebook checks this condition only in the controlled synthetic setting.

The term-structure diagnostic is performed at fixed log-moneyness, not fixed raw strike, because the forward price changes with maturity.

---

## Diagnostic principle

The notebook separates three levels of static consistency:

1. **Price-level validity**  
   Individual option prices must satisfy pointwise bounds.

2. **Cross-strike validity**  
   Prices must satisfy monotonicity and convexity across strikes for each fixed maturity.

3. **Cross-maturity validity**  
   Total implied variance must not decrease across maturities at fixed log-moneyness.

A surface can pass one group of diagnostics and fail another. Therefore, every diagnostic should report:

- pass/fail status;
- failure count;
- maximum violation size;
- affected maturity;
- affected strike, strike pair, or strike triplet where applicable;
- severity label.

The notebook should not only report whether a surface fails. It should report where, how badly, and under which no-arbitrage condition.

In [2]:
# ============================================================
# Cell 4: Core financial primitives and scalar-safe BSM functions
# ============================================================

def _all_inputs_scalar(*values) -> bool:
    """
    Return True when every input is scalar-like.

    This lets the pricing functions support both:

        float output for scalar inputs
        ndarray output for vector inputs

    without breaking boolean indexing on zero-dimensional arrays.
    """
    return all(np.ndim(value) == 0 for value in values)


def _as_1d_float_array(value) -> np.ndarray:
    """
    Convert scalar/list/array input into a one-dimensional float array.

    Using at least_1d avoids the common scalar bug where a numpy.float64
    cannot be indexed or assigned into.
    """
    return np.atleast_1d(np.asarray(value, dtype=float))


def _maybe_scalar(output: np.ndarray, scalar_output: bool):
    """
    Return a Python float for scalar input, otherwise return an ndarray.
    """
    output = np.asarray(output, dtype=float)
    if scalar_output:
        return float(output.reshape(-1)[0])
    return output


def discount_factor(T, r: float = r):
    """
    Risk-free discount factor:

        D(T) = exp(-rT)
    """
    scalar_output = _all_inputs_scalar(T)
    T_arr = _as_1d_float_array(T)

    out = np.exp(-r * T_arr)
    return _maybe_scalar(out, scalar_output)


def dividend_discount_factor(T, q: float = q):
    """
    Dividend discount factor:

        Q(T) = exp(-qT)
    """
    scalar_output = _all_inputs_scalar(T)
    T_arr = _as_1d_float_array(T)

    out = np.exp(-q * T_arr)
    return _maybe_scalar(out, scalar_output)


def forward_price(S0, T, r: float = r, q: float = q):
    """
    Maturity-specific forward price under deterministic rates and dividends:

        F(T) = S0 * exp((r - q)T)
    """
    scalar_output = _all_inputs_scalar(S0, T)

    S_arr, T_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(T),
    )

    out = S_arr * np.exp((r - q) * T_arr)
    return _maybe_scalar(out, scalar_output)


def prepaid_forward_value(S0, T, q: float = q):
    """
    Prepaid forward value of the stock:

        S0 * exp(-qT)
    """
    scalar_output = _all_inputs_scalar(S0, T)

    S_arr, T_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(T),
    )

    out = S_arr * np.exp(-q * T_arr)
    return _maybe_scalar(out, scalar_output)


def strike_from_log_moneyness(S0, T, k_log, r: float = r, q: float = q):
    """
    Generate strike from maturity-specific forward log-moneyness:

        K = F(T) * exp(k_log)
    """
    scalar_output = _all_inputs_scalar(S0, T, k_log)

    F_arr, k_arr = np.broadcast_arrays(
        _as_1d_float_array(forward_price(S0, T, r=r, q=q)),
        _as_1d_float_array(k_log),
    )

    out = F_arr * np.exp(k_arr)
    return _maybe_scalar(out, scalar_output)


def log_moneyness_from_strike(S0, K, T, r: float = r, q: float = q):
    """
    Compute forward log-moneyness:

        k = log(K / F(T))
    """
    scalar_output = _all_inputs_scalar(S0, K, T)

    K_arr, F_arr = np.broadcast_arrays(
        _as_1d_float_array(K),
        _as_1d_float_array(forward_price(S0, T, r=r, q=q)),
    )

    out = np.log(K_arr / F_arr)
    return _maybe_scalar(out, scalar_output)


def bsm_d1_d2(S0, K, T, sigma, r: float = r, q: float = q):
    """
    Black-Scholes-Merton d1 and d2 for a dividend-paying underlying.

    The function is vectorized and scalar-safe. It expects strictly positive
    S0, K, T, and sigma for active pricing use.
    """
    scalar_output = _all_inputs_scalar(S0, K, T, sigma)

    S_arr, K_arr, T_arr, sig_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
        _as_1d_float_array(sigma),
    )

    if np.any(S_arr <= 0):
        raise ValueError("S0 must be strictly positive.")
    if np.any(K_arr <= 0):
        raise ValueError("K must be strictly positive.")
    if np.any(T_arr <= 0):
        raise ValueError("T must be strictly positive for d1/d2.")
    if np.any(sig_arr <= 0):
        raise ValueError("sigma must be strictly positive for d1/d2.")

    sqrt_T = np.sqrt(T_arr)

    d1 = (
        np.log(S_arr / K_arr)
        + (r - q + 0.5 * sig_arr**2) * T_arr
    ) / (sig_arr * sqrt_T)

    d2 = d1 - sig_arr * sqrt_T

    if scalar_output:
        return float(d1[0]), float(d2[0])

    return d1, d2


def bsm_call_price(S0, K, T, sigma, r: float = r, q: float = q):
    """
    Black-Scholes-Merton European call price with continuous dividend yield.

    Handles scalar and vector inputs safely.

    For expired options or zero volatility, the function returns discounted
    forward intrinsic value:

        D(T) * max(F(T) - K, 0)
    """
    scalar_output = _all_inputs_scalar(S0, K, T, sigma)

    S_arr, K_arr, T_arr, sig_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
        _as_1d_float_array(sigma),
    )

    if np.any(S_arr <= 0):
        raise ValueError("S0 must be strictly positive.")
    if np.any(K_arr <= 0):
        raise ValueError("K must be strictly positive.")
    if np.any(T_arr < 0):
        raise ValueError("T must be nonnegative.")
    if np.any(sig_arr < 0):
        raise ValueError("sigma must be nonnegative.")

    D_arr = np.exp(-r * T_arr)
    F_arr = S_arr * np.exp((r - q) * T_arr)

    out = D_arr * np.maximum(F_arr - K_arr, 0.0)

    active = (T_arr > 0.0) & (sig_arr > 0.0)

    if np.any(active):
        d1, d2 = bsm_d1_d2(
            S0=S_arr[active],
            K=K_arr[active],
            T=T_arr[active],
            sigma=sig_arr[active],
            r=r,
            q=q,
        )

        out[active] = (
            S_arr[active] * np.exp(-q * T_arr[active]) * norm.cdf(d1)
            - K_arr[active] * np.exp(-r * T_arr[active]) * norm.cdf(d2)
        )

    return _maybe_scalar(out, scalar_output)


def bsm_put_price(S0, K, T, sigma, r: float = r, q: float = q):
    """
    Black-Scholes-Merton European put price with continuous dividend yield.

    Handles scalar and vector inputs safely.

    For expired options or zero volatility, the function returns discounted
    forward intrinsic value:

        D(T) * max(K - F(T), 0)
    """
    scalar_output = _all_inputs_scalar(S0, K, T, sigma)

    S_arr, K_arr, T_arr, sig_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
        _as_1d_float_array(sigma),
    )

    if np.any(S_arr <= 0):
        raise ValueError("S0 must be strictly positive.")
    if np.any(K_arr <= 0):
        raise ValueError("K must be strictly positive.")
    if np.any(T_arr < 0):
        raise ValueError("T must be nonnegative.")
    if np.any(sig_arr < 0):
        raise ValueError("sigma must be nonnegative.")

    D_arr = np.exp(-r * T_arr)
    F_arr = S_arr * np.exp((r - q) * T_arr)

    out = D_arr * np.maximum(K_arr - F_arr, 0.0)

    active = (T_arr > 0.0) & (sig_arr > 0.0)

    if np.any(active):
        d1, d2 = bsm_d1_d2(
            S0=S_arr[active],
            K=K_arr[active],
            T=T_arr[active],
            sigma=sig_arr[active],
            r=r,
            q=q,
        )

        out[active] = (
            K_arr[active] * np.exp(-r * T_arr[active]) * norm.cdf(-d2)
            - S_arr[active] * np.exp(-q * T_arr[active]) * norm.cdf(-d1)
        )

    return _maybe_scalar(out, scalar_output)


def european_option_bounds(S0, K, T, r: float = r, q: float = q) -> pd.DataFrame:
    """
    Return European call and put pointwise no-arbitrage bounds.

    Bounds:

        call_lower = D(T) * max(F(T) - K, 0)
        call_upper = D(T) * F(T)

        put_lower  = D(T) * max(K - F(T), 0)
        put_upper  = D(T) * K
    """
    S_arr, K_arr, T_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
    )

    if np.any(S_arr <= 0):
        raise ValueError("S0 must be strictly positive.")
    if np.any(K_arr <= 0):
        raise ValueError("K must be strictly positive.")
    if np.any(T_arr < 0):
        raise ValueError("T must be nonnegative.")

    D_arr = np.exp(-r * T_arr)
    F_arr = S_arr * np.exp((r - q) * T_arr)

    out = pd.DataFrame(
        {
            "S0": S_arr,
            "K": K_arr,
            "T": T_arr,
            "D": D_arr,
            "F": F_arr,
            "call_lower": D_arr * np.maximum(F_arr - K_arr, 0.0),
            "call_upper": D_arr * F_arr,
            "put_lower": D_arr * np.maximum(K_arr - F_arr, 0.0),
            "put_upper": D_arr * K_arr,
        }
    )

    return out


def put_call_parity_residual(call_price, put_price, S0, K, T, r: float = r, q: float = q):
    """
    Put-call parity residual:

        residual = C - P - D(T) * (F(T) - K)

    A clean synthetic surface should produce residuals near zero.
    """
    scalar_output = _all_inputs_scalar(call_price, put_price, S0, K, T)

    C_arr, P_arr, K_arr, T_arr, F_arr, D_arr = np.broadcast_arrays(
        _as_1d_float_array(call_price),
        _as_1d_float_array(put_price),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
        _as_1d_float_array(forward_price(S0, T, r=r, q=q)),
        _as_1d_float_array(discount_factor(T, r=r)),
    )

    out = C_arr - P_arr - D_arr * (F_arr - K_arr)
    return _maybe_scalar(out, scalar_output)


# ------------------------------------------------------------
# Scalar and vector sanity checks
# ------------------------------------------------------------

sample_T = 0.50
sample_k_log = 0.00
sample_K = strike_from_log_moneyness(S0=S0, T=sample_T, k_log=sample_k_log, r=r, q=q)

sample_call = bsm_call_price(S0=S0, K=sample_K, T=sample_T, sigma=BASE_VOL, r=r, q=q)
sample_put = bsm_put_price(S0=S0, K=sample_K, T=sample_T, sigma=BASE_VOL, r=r, q=q)
sample_bounds = european_option_bounds(S0=S0, K=sample_K, T=sample_T, r=r, q=q)
sample_parity_resid = put_call_parity_residual(
    call_price=sample_call,
    put_price=sample_put,
    S0=S0,
    K=sample_K,
    T=sample_T,
    r=r,
    q=q,
)

vector_K = strike_from_log_moneyness(S0=S0, T=sample_T, k_log=K_LOG_GRID, r=r, q=q)
vector_calls = bsm_call_price(S0=S0, K=vector_K, T=sample_T, sigma=BASE_VOL, r=r, q=q)
vector_puts = bsm_put_price(S0=S0, K=vector_K, T=sample_T, sigma=BASE_VOL, r=r, q=q)
vector_parity = put_call_parity_residual(
    call_price=vector_calls,
    put_price=vector_puts,
    S0=S0,
    K=vector_K,
    T=sample_T,
    r=r,
    q=q,
)

primitive_audit = pd.DataFrame(
    {
        "check": [
            "sample_forward_log_moneyness",
            "sample_strike",
            "sample_call_price",
            "sample_put_price",
            "sample_parity_residual_abs",
            "sample_call_inside_bounds",
            "sample_put_inside_bounds",
            "vector_call_count",
            "vector_put_count",
            "vector_max_abs_parity_residual",
        ],
        "value": [
            log_moneyness_from_strike(S0=S0, K=sample_K, T=sample_T, r=r, q=q),
            sample_K,
            sample_call,
            sample_put,
            abs(sample_parity_resid),
            (
                sample_bounds["call_lower"].iloc[0] - TOL.price_abs
                <= sample_call
                <= sample_bounds["call_upper"].iloc[0] + TOL.price_abs
            ),
            (
                sample_bounds["put_lower"].iloc[0] - TOL.price_abs
                <= sample_put
                <= sample_bounds["put_upper"].iloc[0] + TOL.price_abs
            ),
            len(vector_calls),
            len(vector_puts),
            float(np.max(np.abs(vector_parity))),
        ],
    }
)

primitive_audit

,check,value
0,sample_forward_log_moneyness,0.0000000000
1,sample_strike,101.0050167084
2,sample_call_price,5.6090821385
3,sample_put_price,5.6090821385
4,sample_parity_residual_abs,0.0000000000
5,sample_call_inside_bounds,True
6,sample_put_inside_bounds,True
7,vector_call_count,11
8,vector_put_count,11
9,vector_max_abs_parity_residual,0.0000000000


In [3]:
# ============================================================
# Cell 5: Build the clean synthetic Black-Scholes benchmark surface
# ============================================================

def synthetic_vol_surface(k_log, T, base_vol: float = BASE_VOL):
    """
    Controlled volatility surface used for the clean positive-control benchmark.

    For the first benchmark, use a constant volatility surface:

        sigma(k, T) = base_vol

    This keeps the positive control simple. Later negative controls will corrupt
    prices or implied volatilities deliberately.
    """
    scalar_output = _all_inputs_scalar(k_log, T)

    k_arr, T_arr = np.broadcast_arrays(
        _as_1d_float_array(k_log),
        _as_1d_float_array(T),
    )

    out = np.full_like(k_arr, fill_value=base_vol, dtype=float)
    return _maybe_scalar(out, scalar_output)


def build_clean_bsm_surface(
    S0: float = S0,
    maturities: np.ndarray = MATURITIES,
    k_log_grid: np.ndarray = K_LOG_GRID,
    base_vol: float = BASE_VOL,
    r: float = r,
    q: float = q,
) -> pd.DataFrame:
    """
    Build a clean synthetic European option surface.

    The surface is generated on a fixed log-moneyness grid. For each maturity T:

        F(T) = S0 * exp((r - q)T)
        K    = F(T) * exp(k_log)

    Then Black-Scholes-Merton call and put prices are generated using the
    controlled volatility surface.

    Returns a wide surface table with one row per (T, k_log).
    """
    rows = []

    for T_value in maturities:
        F_value = forward_price(S0=S0, T=T_value, r=r, q=q)
        D_value = discount_factor(T=T_value, r=r)

        for k_value in k_log_grid:
            K_value = strike_from_log_moneyness(
                S0=S0,
                T=T_value,
                k_log=k_value,
                r=r,
                q=q,
            )

            sigma_value = synthetic_vol_surface(
                k_log=k_value,
                T=T_value,
                base_vol=base_vol,
            )

            call_value = bsm_call_price(
                S0=S0,
                K=K_value,
                T=T_value,
                sigma=sigma_value,
                r=r,
                q=q,
            )

            put_value = bsm_put_price(
                S0=S0,
                K=K_value,
                T=T_value,
                sigma=sigma_value,
                r=r,
                q=q,
            )

            bounds = european_option_bounds(
                S0=S0,
                K=K_value,
                T=T_value,
                r=r,
                q=q,
            ).iloc[0]

            parity_resid = put_call_parity_residual(
                call_price=call_value,
                put_price=put_value,
                S0=S0,
                K=K_value,
                T=T_value,
                r=r,
                q=q,
            )

            rows.append(
                {
                    "surface_id": "clean_bsm_constant_vol",
                    "S0": float(S0),
                    "r": float(r),
                    "q": float(q),
                    "T": float(T_value),
                    "k_log": float(k_value),
                    "F": float(F_value),
                    "D": float(D_value),
                    "K": float(K_value),
                    "sigma_true": float(sigma_value),
                    "total_variance_true": float(sigma_value**2 * T_value),
                    "call_price": float(call_value),
                    "put_price": float(put_value),
                    "call_lower": float(bounds["call_lower"]),
                    "call_upper": float(bounds["call_upper"]),
                    "put_lower": float(bounds["put_lower"]),
                    "put_upper": float(bounds["put_upper"]),
                    "parity_residual": float(parity_resid),
                }
            )

    surface = pd.DataFrame(rows)

    surface = surface.sort_values(["T", "K"]).reset_index(drop=True)

    surface["call_time_value"] = surface["call_price"] - surface["call_lower"]
    surface["put_time_value"] = surface["put_price"] - surface["put_lower"]

    surface["call_bound_lower_slack"] = surface["call_price"] - surface["call_lower"]
    surface["call_bound_upper_slack"] = surface["call_upper"] - surface["call_price"]
    surface["put_bound_lower_slack"] = surface["put_price"] - surface["put_lower"]
    surface["put_bound_upper_slack"] = surface["put_upper"] - surface["put_price"]

    return surface


clean_surface = build_clean_bsm_surface(
    S0=S0,
    maturities=MATURITIES,
    k_log_grid=K_LOG_GRID,
    base_vol=BASE_VOL,
    r=r,
    q=q,
)


# ------------------------------------------------------------
# Positive-control surface audit
# ------------------------------------------------------------

expected_rows = len(MATURITIES) * len(K_LOG_GRID)

clean_surface_audit = pd.DataFrame(
    {
        "check": [
            "surface_id",
            "row_count",
            "expected_row_count",
            "maturity_count",
            "expected_maturity_count",
            "log_moneyness_count",
            "expected_log_moneyness_count",
            "min_strike",
            "max_strike",
            "min_call_price",
            "max_call_price",
            "min_put_price",
            "max_put_price",
            "min_call_lower_slack",
            "min_call_upper_slack",
            "min_put_lower_slack",
            "min_put_upper_slack",
            "max_abs_parity_residual",
            "min_total_variance",
            "max_total_variance",
        ],
        "value": [
            clean_surface["surface_id"].iloc[0],
            len(clean_surface),
            expected_rows,
            clean_surface["T"].nunique(),
            len(MATURITIES),
            clean_surface["k_log"].nunique(),
            len(K_LOG_GRID),
            clean_surface["K"].min(),
            clean_surface["K"].max(),
            clean_surface["call_price"].min(),
            clean_surface["call_price"].max(),
            clean_surface["put_price"].min(),
            clean_surface["put_price"].max(),
            clean_surface["call_bound_lower_slack"].min(),
            clean_surface["call_bound_upper_slack"].min(),
            clean_surface["put_bound_lower_slack"].min(),
            clean_surface["put_bound_upper_slack"].min(),
            clean_surface["parity_residual"].abs().max(),
            clean_surface["total_variance_true"].min(),
            clean_surface["total_variance_true"].max(),
        ],
    }
)

clean_surface_audit

,check,value
0,surface_id,clean_bsm_constant_vol
1,row_count,55
2,expected_row_count,55
3,maturity_count,5
4,expected_maturity_count,5
5,log_moneyness_count,11
6,expected_log_moneyness_count,11
7,min_strike,67.1662027662
8,max_strike,155.2707218511
9,min_call_price,0.0000000001


In [4]:
# ============================================================
# Cell 6: Diagnostic record schema, validation helpers, and severity logic
# ============================================================

BASE_SURFACE_COLUMNS = [
    "surface_id",
    "S0",
    "r",
    "q",
    "T",
    "k_log",
    "F",
    "D",
    "K",
    "sigma_true",
    "total_variance_true",
    "call_price",
    "put_price",
    "call_lower",
    "call_upper",
    "put_lower",
    "put_upper",
    "parity_residual",
]


DIAGNOSTIC_RECORD_COLUMNS = [
    "surface_id",
    "diagnostic_group",
    "check_name",
    "status",
    "failure_count",
    "max_violation",
    "tolerance",
    "severity",
    "affected_T",
    "affected_K",
    "affected_k_log",
    "affected_lower_K",
    "affected_middle_K",
    "affected_upper_K",
    "message",
]


def require_columns(df: pd.DataFrame, required_columns: Iterable[str], table_name: str = "DataFrame") -> None:
    """
    Raise a clear error if a table is missing required columns.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            f"{table_name} is missing required columns: {missing}"
        )


def status_from_failures(failure_count: int) -> str:
    """
    Convert a failure count into PASS / FAIL status.
    """
    return STATUS_PASS if int(failure_count) == 0 else STATUS_FAIL


def max_abs_or_zero(values) -> float:
    """
    Return max(abs(values)) as a float.

    Empty inputs return 0.0. NaNs are ignored unless all values are NaN.
    """
    arr = np.asarray(values, dtype=float).reshape(-1)

    if arr.size == 0:
        return 0.0

    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        return 0.0

    return float(np.max(np.abs(finite)))


def max_positive_or_zero(values) -> float:
    """
    Return max(values) over finite positive violations.

    Empty inputs, all-NaN inputs, and nonpositive inputs return 0.0.
    """
    arr = np.asarray(values, dtype=float).reshape(-1)

    if arr.size == 0:
        return 0.0

    finite = arr[np.isfinite(arr)]

    if finite.size == 0:
        return 0.0

    positive = finite[finite > 0.0]

    if positive.size == 0:
        return 0.0

    return float(np.max(positive))


def severity_from_violation(
    failure_count: int,
    max_violation: float,
    tolerance: float,
    *,
    clean_benchmark: bool = False,
    critical_on_fail: bool = False,
) -> str:
    """
    Assign a severity label for a diagnostic record.

    For this synthetic notebook:
    - zero failures are INFO;
    - clean benchmark failures can be escalated because they imply implementation error;
    - otherwise severity is based on violation magnitude relative to tolerance.
    """
    failure_count = int(failure_count)
    max_violation = float(abs(max_violation))
    tolerance = float(abs(tolerance))

    if failure_count == 0:
        return SEVERITY_INFO

    if critical_on_fail:
        return SEVERITY_CRITICAL

    if clean_benchmark:
        return SEVERITY_CRITICAL

    scale = max(tolerance, 1.0e-16)
    ratio = max_violation / scale

    if ratio <= 10.0:
        return SEVERITY_LOW
    if ratio <= 1.0e4:
        return SEVERITY_MEDIUM

    return SEVERITY_HIGH


def first_affected_value(failures: pd.DataFrame, column: str):
    """
    Extract the first affected value from a failure table.

    Returns np.nan if the column is unavailable or the table is empty.
    """
    if failures is None or len(failures) == 0 or column not in failures.columns:
        return np.nan

    value = failures[column].iloc[0]

    if isinstance(value, (np.integer, np.floating)):
        return float(value)

    return value


def make_diagnostic_record(
    *,
    surface_id: str,
    diagnostic_group: str,
    check_name: str,
    failure_count: int,
    max_violation: float,
    tolerance: float,
    severity: Optional[str] = None,
    affected_T=np.nan,
    affected_K=np.nan,
    affected_k_log=np.nan,
    affected_lower_K=np.nan,
    affected_middle_K=np.nan,
    affected_upper_K=np.nan,
    message: str = "",
    clean_benchmark: bool = False,
    critical_on_fail: bool = False,
) -> Dict[str, object]:
    """
    Build one standardized diagnostic record.

    Every diagnostic should report:
    - pass/fail status;
    - failure count;
    - maximum violation size;
    - tolerance used;
    - severity;
    - affected location if applicable.
    """
    failure_count = int(failure_count)
    max_violation = float(max_violation)
    tolerance = float(tolerance)

    status = status_from_failures(failure_count)

    if severity is None:
        severity = severity_from_violation(
            failure_count=failure_count,
            max_violation=max_violation,
            tolerance=tolerance,
            clean_benchmark=clean_benchmark,
            critical_on_fail=critical_on_fail,
        )

    return {
        "surface_id": surface_id,
        "diagnostic_group": diagnostic_group,
        "check_name": check_name,
        "status": status,
        "failure_count": failure_count,
        "max_violation": max_violation,
        "tolerance": tolerance,
        "severity": severity,
        "affected_T": affected_T,
        "affected_K": affected_K,
        "affected_k_log": affected_k_log,
        "affected_lower_K": affected_lower_K,
        "affected_middle_K": affected_middle_K,
        "affected_upper_K": affected_upper_K,
        "message": message,
    }


def diagnostic_records_to_frame(records: List[Dict[str, object]]) -> pd.DataFrame:
    """
    Convert diagnostic records into a stable ledger DataFrame.
    """
    if len(records) == 0:
        return pd.DataFrame(columns=DIAGNOSTIC_RECORD_COLUMNS)

    ledger = pd.DataFrame(records)

    for col in DIAGNOSTIC_RECORD_COLUMNS:
        if col not in ledger.columns:
            ledger[col] = np.nan

    return ledger[DIAGNOSTIC_RECORD_COLUMNS].copy()


def summarize_diagnostic_ledger(ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize a diagnostic ledger by group.
    """
    require_columns(
        ledger,
        ["diagnostic_group", "status", "failure_count", "max_violation", "severity"],
        table_name="diagnostic ledger",
    )

    if len(ledger) == 0:
        return pd.DataFrame(
            columns=[
                "diagnostic_group",
                "checks",
                "failed_checks",
                "total_failures",
                "max_violation",
                "worst_severity",
            ]
        )

    severity_rank = {
        SEVERITY_INFO: 0,
        SEVERITY_LOW: 1,
        SEVERITY_MEDIUM: 2,
        SEVERITY_HIGH: 3,
        SEVERITY_CRITICAL: 4,
    }

    tmp = ledger.copy()
    tmp["_severity_rank"] = tmp["severity"].map(severity_rank).fillna(-1).astype(int)

    grouped = (
        tmp.groupby("diagnostic_group", as_index=False)
        .agg(
            checks=("check_name", "count"),
            failed_checks=("status", lambda x: int((x == STATUS_FAIL).sum())),
            total_failures=("failure_count", "sum"),
            max_violation=("max_violation", "max"),
            worst_severity_rank=("_severity_rank", "max"),
        )
    )

    inverse_severity_rank = {value: key for key, value in severity_rank.items()}
    grouped["worst_severity"] = grouped["worst_severity_rank"].map(inverse_severity_rank)

    grouped = grouped.drop(columns=["worst_severity_rank"])

    return grouped


def assert_clean_surface_contract(surface: pd.DataFrame) -> None:
    """
    Validate that the clean synthetic surface has the required structure.
    """
    require_columns(
        surface,
        BASE_SURFACE_COLUMNS,
        table_name="clean synthetic surface",
    )

    if len(surface) == 0:
        raise ValueError("clean synthetic surface is empty.")

    if surface["T"].nunique() != len(MATURITIES):
        raise ValueError("clean synthetic surface has unexpected maturity count.")

    if surface["k_log"].nunique() != len(K_LOG_GRID):
        raise ValueError("clean synthetic surface has unexpected log-moneyness count.")

    if not np.all(np.isfinite(surface["call_price"])):
        raise ValueError("clean synthetic surface has non-finite call prices.")

    if not np.all(np.isfinite(surface["put_price"])):
        raise ValueError("clean synthetic surface has non-finite put prices.")

    if np.any(surface["K"] <= 0):
        raise ValueError("clean synthetic surface has nonpositive strikes.")

    if np.any(surface["T"] <= 0):
        raise ValueError("clean synthetic surface has nonpositive maturities.")


# ------------------------------------------------------------
# Contract audit for the diagnostic framework
# ------------------------------------------------------------

assert_clean_surface_contract(clean_surface)

schema_audit_records = [
    {
        "object": "BASE_SURFACE_COLUMNS",
        "count": len(BASE_SURFACE_COLUMNS),
        "status": STATUS_PASS,
        "message": "Required synthetic surface columns are defined.",
    },
    {
        "object": "DIAGNOSTIC_RECORD_COLUMNS",
        "count": len(DIAGNOSTIC_RECORD_COLUMNS),
        "status": STATUS_PASS,
        "message": "Standard diagnostic ledger schema is defined.",
    },
    {
        "object": "clean_surface_contract",
        "count": len(clean_surface),
        "status": STATUS_PASS,
        "message": "Clean surface satisfies required structural contract.",
    },
    {
        "object": "severity_labels",
        "count": 5,
        "status": STATUS_PASS,
        "message": ", ".join(
            [
                SEVERITY_INFO,
                SEVERITY_LOW,
                SEVERITY_MEDIUM,
                SEVERITY_HIGH,
                SEVERITY_CRITICAL,
            ]
        ),
    },
    {
        "object": "status_labels",
        "count": 2,
        "status": STATUS_PASS,
        "message": ", ".join([STATUS_PASS, STATUS_FAIL]),
    },
]

diagnostic_framework_audit = pd.DataFrame(schema_audit_records)

diagnostic_framework_audit

,object,count,status,message
0,BASE_SURFACE_COLUMNS,18,PASS,Required synthetic surface columns are defined.
1,DIAGNOSTIC_RECORD_COLUMNS,15,PASS,Standard diagnostic ledger schema is defined.
2,clean_surface_contract,55,PASS,Clean surface satisfies required structural co...
3,severity_labels,5,PASS,"INFO, LOW, MEDIUM, HIGH, CRITICAL"
4,status_labels,2,PASS,"PASS, FAIL"


In [5]:
# ============================================================
# Cell 7: Pointwise option bounds diagnostics
# ============================================================

POINTWISE_REQUIRED_COLUMNS = [
    "surface_id",
    "T",
    "k_log",
    "K",
    "call_price",
    "put_price",
    "call_lower",
    "call_upper",
    "put_lower",
    "put_upper",
]


def _worst_failure_row(failures: pd.DataFrame, violation_col: str = "violation") -> pd.Series:
    """
    Return the row with the largest violation.

    If the failure table is empty, return an empty Series.
    """
    if failures is None or len(failures) == 0:
        return pd.Series(dtype=object)

    idx = failures[violation_col].astype(float).idxmax()
    return failures.loc[idx]


def _build_pointwise_failure_table(
    surface: pd.DataFrame,
    *,
    check_name: str,
    option_type: str,
    bound_side: str,
    price_col: str,
    bound_col: str,
    violation_col: str,
    tolerance: float,
) -> pd.DataFrame:
    """
    Build a standardized failure detail table for one pointwise bound check.

    A positive violation means the no-arbitrage inequality is breached.

    Lower-bound check:

        violation = lower_bound - price

    Upper-bound check:

        violation = price - upper_bound
    """
    tmp = surface.copy()

    tmp["diagnostic_group"] = "pointwise_bounds"
    tmp["check_name"] = check_name
    tmp["option_type"] = option_type
    tmp["bound_side"] = bound_side
    tmp["price"] = tmp[price_col].astype(float)
    tmp["bound"] = tmp[bound_col].astype(float)
    tmp["violation"] = tmp[violation_col].astype(float)
    tmp["tolerance"] = float(tolerance)
    tmp["is_failure"] = tmp["violation"] > float(tolerance)

    failure_cols = [
        "surface_id",
        "diagnostic_group",
        "check_name",
        "option_type",
        "bound_side",
        "T",
        "k_log",
        "K",
        "price",
        "bound",
        "violation",
        "tolerance",
        "is_failure",
    ]

    return tmp.loc[tmp["is_failure"], failure_cols].reset_index(drop=True)


def diagnose_pointwise_bounds(
    surface: pd.DataFrame,
    *,
    tolerance: float = TOL.price_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Diagnose pointwise European option bound violations.

    Checks:

    1. call lower bound:
           C(K,T) >= D(T) max(F(T) - K, 0)

    2. call upper bound:
           C(K,T) <= D(T) F(T)

    3. put lower bound:
           P(K,T) >= D(T) max(K - F(T), 0)

    4. put upper bound:
           P(K,T) <= D(T) K

    Returns:
        ledger:
            one standardized diagnostic record per check

        failure_details:
            row-level failures with affected maturity, strike, and violation size
    """
    require_columns(
        surface,
        POINTWISE_REQUIRED_COLUMNS,
        table_name="surface for pointwise bounds diagnostics",
    )

    if len(surface) == 0:
        raise ValueError("surface for pointwise bounds diagnostics is empty.")

    surface_id = str(surface["surface_id"].iloc[0])

    work = surface.copy()

    work["call_lower_violation"] = work["call_lower"].astype(float) - work["call_price"].astype(float)
    work["call_upper_violation"] = work["call_price"].astype(float) - work["call_upper"].astype(float)

    work["put_lower_violation"] = work["put_lower"].astype(float) - work["put_price"].astype(float)
    work["put_upper_violation"] = work["put_price"].astype(float) - work["put_upper"].astype(float)

    check_specs = [
        {
            "check_name": "call_lower_bound",
            "option_type": "call",
            "bound_side": "lower",
            "price_col": "call_price",
            "bound_col": "call_lower",
            "violation_col": "call_lower_violation",
            "message_pass": "All call prices are above their lower no-arbitrage bound.",
            "message_fail": "At least one call price is below its lower no-arbitrage bound.",
        },
        {
            "check_name": "call_upper_bound",
            "option_type": "call",
            "bound_side": "upper",
            "price_col": "call_price",
            "bound_col": "call_upper",
            "violation_col": "call_upper_violation",
            "message_pass": "All call prices are below their upper no-arbitrage bound.",
            "message_fail": "At least one call price is above its upper no-arbitrage bound.",
        },
        {
            "check_name": "put_lower_bound",
            "option_type": "put",
            "bound_side": "lower",
            "price_col": "put_price",
            "bound_col": "put_lower",
            "violation_col": "put_lower_violation",
            "message_pass": "All put prices are above their lower no-arbitrage bound.",
            "message_fail": "At least one put price is below its lower no-arbitrage bound.",
        },
        {
            "check_name": "put_upper_bound",
            "option_type": "put",
            "bound_side": "upper",
            "price_col": "put_price",
            "bound_col": "put_upper",
            "violation_col": "put_upper_violation",
            "message_pass": "All put prices are below their upper no-arbitrage bound.",
            "message_fail": "At least one put price is above its upper no-arbitrage bound.",
        },
    ]

    records = []
    failure_tables = []

    for spec in check_specs:
        failures = _build_pointwise_failure_table(
            work,
            check_name=spec["check_name"],
            option_type=spec["option_type"],
            bound_side=spec["bound_side"],
            price_col=spec["price_col"],
            bound_col=spec["bound_col"],
            violation_col=spec["violation_col"],
            tolerance=tolerance,
        )

        failure_tables.append(failures)

        failure_count = int(len(failures))
        max_violation = max_positive_or_zero(work[spec["violation_col"]])

        worst = _worst_failure_row(failures, violation_col="violation")

        records.append(
            make_diagnostic_record(
                surface_id=surface_id,
                diagnostic_group="pointwise_bounds",
                check_name=spec["check_name"],
                failure_count=failure_count,
                max_violation=max_violation,
                tolerance=tolerance,
                affected_T=worst.get("T", np.nan),
                affected_K=worst.get("K", np.nan),
                affected_k_log=worst.get("k_log", np.nan),
                message=spec["message_pass"] if failure_count == 0 else spec["message_fail"],
                clean_benchmark=clean_benchmark,
            )
        )

    ledger = diagnostic_records_to_frame(records)

    if len(failure_tables) > 0:
        failure_details = pd.concat(failure_tables, ignore_index=True)
    else:
        failure_details = pd.DataFrame()

    return ledger, failure_details


# ------------------------------------------------------------
# Run pointwise bounds diagnostics on the clean positive-control surface
# ------------------------------------------------------------

clean_pointwise_ledger, clean_pointwise_failures = diagnose_pointwise_bounds(
    clean_surface,
    tolerance=TOL.price_abs,
    clean_benchmark=True,
)

clean_pointwise_summary = summarize_diagnostic_ledger(clean_pointwise_ledger)

clean_pointwise_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,pointwise_bounds,call_lower_bound,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call prices are above their lower no-arbit...
1,clean_bsm_constant_vol,pointwise_bounds,call_upper_bound,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call prices are below their upper no-arbit...
2,clean_bsm_constant_vol,pointwise_bounds,put_lower_bound,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put prices are above their lower no-arbitr...
3,clean_bsm_constant_vol,pointwise_bounds,put_upper_bound,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put prices are below their upper no-arbitr...


In [6]:
# ============================================================
# Cell 8: Put-call parity diagnostics
# ============================================================

PARITY_REQUIRED_COLUMNS = [
    "surface_id",
    "S0",
    "T",
    "k_log",
    "K",
    "F",
    "D",
    "call_price",
    "put_price",
]


def diagnose_put_call_parity(
    surface: pd.DataFrame,
    *,
    tolerance: float = TOL.parity_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Diagnose European put-call parity consistency.

    For deterministic rates and proportional dividends:

        C(K,T) - P(K,T) = D(T) * (F(T) - K)

    The diagnostic recomputes the residual instead of trusting any preexisting
    parity_residual column.

    Returns:
        ledger:
            one standardized diagnostic record

        failure_details:
            row-level parity failures
    """
    require_columns(
        surface,
        PARITY_REQUIRED_COLUMNS,
        table_name="surface for put-call parity diagnostics",
    )

    if len(surface) == 0:
        raise ValueError("surface for put-call parity diagnostics is empty.")

    surface_id = str(surface["surface_id"].iloc[0])

    work = surface.copy()

    work["parity_target"] = work["D"].astype(float) * (
        work["F"].astype(float) - work["K"].astype(float)
    )

    work["parity_residual_recomputed"] = (
        work["call_price"].astype(float)
        - work["put_price"].astype(float)
        - work["parity_target"].astype(float)
    )

    work["parity_abs_residual"] = work["parity_residual_recomputed"].abs()
    work["violation"] = work["parity_abs_residual"] - float(tolerance)
    work["is_failure"] = work["violation"] > 0.0

    failure_cols = [
        "surface_id",
        "T",
        "k_log",
        "K",
        "F",
        "D",
        "call_price",
        "put_price",
        "parity_target",
        "parity_residual_recomputed",
        "parity_abs_residual",
        "violation",
        "is_failure",
    ]

    failure_details = (
        work.loc[work["is_failure"], failure_cols]
        .sort_values("parity_abs_residual", ascending=False)
        .reset_index(drop=True)
    )

    failure_count = int(len(failure_details))
    max_violation = float(work["parity_abs_residual"].max())

    if failure_count > 0:
        worst = failure_details.iloc[0]
    else:
        worst = pd.Series(dtype=object)

    record = make_diagnostic_record(
        surface_id=surface_id,
        diagnostic_group="put_call_parity",
        check_name="put_call_parity_residual",
        failure_count=failure_count,
        max_violation=max_violation,
        tolerance=tolerance,
        affected_T=worst.get("T", np.nan),
        affected_K=worst.get("K", np.nan),
        affected_k_log=worst.get("k_log", np.nan),
        message=(
            "All call-put pairs satisfy put-call parity within tolerance."
            if failure_count == 0
            else "At least one call-put pair violates put-call parity."
        ),
        clean_benchmark=clean_benchmark,
    )

    ledger = diagnostic_records_to_frame([record])

    return ledger, failure_details


# ------------------------------------------------------------
# Run put-call parity diagnostics on the clean positive-control surface
# ------------------------------------------------------------

clean_parity_ledger, clean_parity_failures = diagnose_put_call_parity(
    clean_surface,
    tolerance=TOL.parity_abs,
    clean_benchmark=True,
)

clean_parity_summary = summarize_diagnostic_ledger(clean_parity_ledger)

clean_parity_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,put_call_parity,put_call_parity_residual,PASS,0,0.0000000000,0.0000000010,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call-put pairs satisfy put-call parity wit...


In [7]:
# ============================================================
# Cell 9: Vertical-spread monotonicity diagnostics
# ============================================================

VERTICAL_REQUIRED_COLUMNS = [
    "surface_id",
    "T",
    "k_log",
    "K",
    "call_price",
    "put_price",
]


def _build_adjacent_strike_pairs(
    surface: pd.DataFrame,
    *,
    option_type: str,
    price_col: str,
) -> pd.DataFrame:
    """
    Build adjacent strike-pair comparisons within each maturity.

    For calls:
        C(K_{i+1}, T) - C(K_i, T) <= 0

    For puts:
        P(K_{i+1}, T) - P(K_i, T) >= 0

    The returned table has one row per adjacent strike pair.
    """
    pair_rows = []

    for T_value, group in surface.groupby("T", sort=True):
        g = group.sort_values("K").reset_index(drop=True)

        if len(g) < 2:
            continue

        for i in range(len(g) - 1):
            left = g.iloc[i]
            right = g.iloc[i + 1]

            left_price = float(left[price_col])
            right_price = float(right[price_col])
            price_diff = right_price - left_price

            pair_rows.append(
                {
                    "surface_id": str(left["surface_id"]),
                    "option_type": option_type,
                    "T": float(T_value),
                    "left_k_log": float(left["k_log"]),
                    "right_k_log": float(right["k_log"]),
                    "left_K": float(left["K"]),
                    "right_K": float(right["K"]),
                    "left_price": left_price,
                    "right_price": right_price,
                    "price_diff_right_minus_left": float(price_diff),
                }
            )

    return pd.DataFrame(pair_rows)


def diagnose_vertical_monotonicity(
    surface: pd.DataFrame,
    *,
    tolerance: float = TOL.monotonicity_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Diagnose vertical-spread monotonicity across strikes.

    For fixed maturity T:

    Calls must be nonincreasing in strike:
        C(K_{i+1}, T) - C(K_i, T) <= 0

    Puts must be nondecreasing in strike:
        P(K_{i+1}, T) - P(K_i, T) >= 0

    Returns:
        ledger:
            one diagnostic record for call monotonicity and one for put monotonicity

        failure_details:
            adjacent strike-pair failures

        pair_table:
            all adjacent strike-pair comparisons
    """
    require_columns(
        surface,
        VERTICAL_REQUIRED_COLUMNS,
        table_name="surface for vertical-spread monotonicity diagnostics",
    )

    if len(surface) == 0:
        raise ValueError("surface for vertical-spread monotonicity diagnostics is empty.")

    surface_id = str(surface["surface_id"].iloc[0])

    call_pairs = _build_adjacent_strike_pairs(
        surface,
        option_type="call",
        price_col="call_price",
    )

    put_pairs = _build_adjacent_strike_pairs(
        surface,
        option_type="put",
        price_col="put_price",
    )

    call_pairs["diagnostic_group"] = "vertical_spread_monotonicity"
    call_pairs["check_name"] = "call_nonincreasing_in_strike"

    put_pairs["diagnostic_group"] = "vertical_spread_monotonicity"
    put_pairs["check_name"] = "put_nondecreasing_in_strike"

    # Calls fail if the higher-strike call is more expensive than the lower-strike call.
    call_pairs["violation"] = call_pairs["price_diff_right_minus_left"]
    call_pairs["tolerance"] = float(tolerance)
    call_pairs["is_failure"] = call_pairs["violation"] > float(tolerance)

    # Puts fail if the higher-strike put is cheaper than the lower-strike put.
    put_pairs["violation"] = -put_pairs["price_diff_right_minus_left"]
    put_pairs["tolerance"] = float(tolerance)
    put_pairs["is_failure"] = put_pairs["violation"] > float(tolerance)

    pair_table = pd.concat([call_pairs, put_pairs], ignore_index=True)

    failure_cols = [
        "surface_id",
        "diagnostic_group",
        "check_name",
        "option_type",
        "T",
        "left_k_log",
        "right_k_log",
        "left_K",
        "right_K",
        "left_price",
        "right_price",
        "price_diff_right_minus_left",
        "violation",
        "tolerance",
        "is_failure",
    ]

    failure_details = (
        pair_table.loc[pair_table["is_failure"], failure_cols]
        .sort_values("violation", ascending=False)
        .reset_index(drop=True)
    )

    records = []

    for check_name, option_type, pairs in [
        ("call_nonincreasing_in_strike", "call", call_pairs),
        ("put_nondecreasing_in_strike", "put", put_pairs),
    ]:
        failures = pairs.loc[pairs["is_failure"]].copy()
        failure_count = int(len(failures))
        max_violation = max_positive_or_zero(pairs["violation"])

        if failure_count > 0:
            worst = failures.sort_values("violation", ascending=False).iloc[0]
        else:
            worst = pd.Series(dtype=object)

        if option_type == "call":
            pass_message = "All adjacent call prices are nonincreasing in strike."
            fail_message = "At least one higher-strike call is more expensive than a lower-strike call."
        else:
            pass_message = "All adjacent put prices are nondecreasing in strike."
            fail_message = "At least one higher-strike put is cheaper than a lower-strike put."

        records.append(
            make_diagnostic_record(
                surface_id=surface_id,
                diagnostic_group="vertical_spread_monotonicity",
                check_name=check_name,
                failure_count=failure_count,
                max_violation=max_violation,
                tolerance=tolerance,
                affected_T=worst.get("T", np.nan),
                affected_K=worst.get("right_K", np.nan),
                affected_k_log=worst.get("right_k_log", np.nan),
                affected_lower_K=worst.get("left_K", np.nan),
                affected_upper_K=worst.get("right_K", np.nan),
                message=pass_message if failure_count == 0 else fail_message,
                clean_benchmark=clean_benchmark,
            )
        )

    ledger = diagnostic_records_to_frame(records)

    return ledger, failure_details, pair_table


# ------------------------------------------------------------
# Run vertical-spread monotonicity diagnostics on the clean positive-control surface
# ------------------------------------------------------------

clean_vertical_ledger, clean_vertical_failures, clean_vertical_pairs = diagnose_vertical_monotonicity(
    clean_surface,
    tolerance=TOL.monotonicity_abs,
    clean_benchmark=True,
)

clean_vertical_summary = summarize_diagnostic_ledger(clean_vertical_ledger)

clean_vertical_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,vertical_spread_monotonicity,call_nonincreasing_in_strike,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All adjacent call prices are nonincreasing in ...
1,clean_bsm_constant_vol,vertical_spread_monotonicity,put_nondecreasing_in_strike,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All adjacent put prices are nondecreasing in s...


In [8]:
# ============================================================
# Cell 10: Butterfly convexity diagnostics
# ============================================================

CONVEXITY_REQUIRED_COLUMNS = [
    "surface_id",
    "T",
    "k_log",
    "K",
    "call_price",
    "put_price",
]


def _build_adjacent_slope_table(
    surface: pd.DataFrame,
    *,
    option_type: str,
    price_col: str,
) -> pd.DataFrame:
    """
    Build adjacent strike slopes within each maturity.

    For a convex function f(K), adjacent slopes must be nondecreasing
    as strike increases.

    Slope on interval i:

        s_i = [f(K_{i+1}) - f(K_i)] / [K_{i+1} - K_i]

    Convexity condition:

        s_{i+1} - s_i >= 0

    This slope-based diagnostic works on uneven strike grids.
    """
    slope_rows = []

    for T_value, group in surface.groupby("T", sort=True):
        g = group.sort_values("K").reset_index(drop=True)

        if len(g) < 2:
            continue

        for i in range(len(g) - 1):
            left = g.iloc[i]
            right = g.iloc[i + 1]

            left_K = float(left["K"])
            right_K = float(right["K"])
            dK = right_K - left_K

            if dK <= 0:
                raise ValueError(
                    f"Non-increasing strike grid detected at T={T_value}: "
                    f"left_K={left_K}, right_K={right_K}"
                )

            left_price = float(left[price_col])
            right_price = float(right[price_col])

            slope = (right_price - left_price) / dK

            slope_rows.append(
                {
                    "surface_id": str(left["surface_id"]),
                    "option_type": option_type,
                    "T": float(T_value),
                    "interval_index": int(i),
                    "left_k_log": float(left["k_log"]),
                    "right_k_log": float(right["k_log"]),
                    "left_K": left_K,
                    "right_K": right_K,
                    "left_price": left_price,
                    "right_price": right_price,
                    "dK": float(dK),
                    "slope": float(slope),
                }
            )

    return pd.DataFrame(slope_rows)


def _build_convexity_triplets_from_slopes(
    slope_table: pd.DataFrame,
    *,
    check_name: str,
    tolerance: float,
) -> pd.DataFrame:
    """
    Convert adjacent slopes into convexity triplet checks.

    Each row compares two adjacent slopes:

        slope_right - slope_left >= 0

    A failure means the option price curve is locally concave over
    the three-strike triplet.
    """
    triplet_rows = []

    if len(slope_table) == 0:
        return pd.DataFrame()

    for (surface_id, option_type, T_value), group in slope_table.groupby(
        ["surface_id", "option_type", "T"],
        sort=True,
    ):
        g = group.sort_values("interval_index").reset_index(drop=True)

        if len(g) < 2:
            continue

        for i in range(len(g) - 1):
            left_interval = g.iloc[i]
            right_interval = g.iloc[i + 1]

            lower_K = float(left_interval["left_K"])
            middle_K = float(left_interval["right_K"])
            upper_K = float(right_interval["right_K"])

            lower_price = float(left_interval["left_price"])
            middle_price = float(left_interval["right_price"])
            upper_price = float(right_interval["right_price"])

            slope_left = float(left_interval["slope"])
            slope_right = float(right_interval["slope"])
            slope_diff = slope_right - slope_left

            violation = -slope_diff

            triplet_rows.append(
                {
                    "surface_id": str(surface_id),
                    "diagnostic_group": "butterfly_convexity",
                    "check_name": check_name,
                    "option_type": option_type,
                    "T": float(T_value),
                    "lower_k_log": float(left_interval["left_k_log"]),
                    "middle_k_log": float(left_interval["right_k_log"]),
                    "upper_k_log": float(right_interval["right_k_log"]),
                    "lower_K": lower_K,
                    "middle_K": middle_K,
                    "upper_K": upper_K,
                    "lower_price": lower_price,
                    "middle_price": middle_price,
                    "upper_price": upper_price,
                    "slope_left": slope_left,
                    "slope_right": slope_right,
                    "slope_diff": float(slope_diff),
                    "violation": float(violation),
                    "tolerance": float(tolerance),
                    "is_failure": bool(violation > tolerance),
                }
            )

    return pd.DataFrame(triplet_rows)


def diagnose_butterfly_convexity(
    surface: pd.DataFrame,
    *,
    tolerance: float = TOL.convexity_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Diagnose butterfly convexity across strikes.

    For fixed maturity T, European call and put prices must be convex in strike.

    The diagnostic uses adjacent strike slopes:

        slope_i = [V(K_{i+1}, T) - V(K_i, T)] / [K_{i+1} - K_i]

    Convexity requires:

        slope_{i+1} - slope_i >= 0

    Returns:
        ledger:
            one diagnostic record for call convexity and one for put convexity

        failure_details:
            failed three-strike triplets

        triplet_table:
            all convexity triplet checks

        slope_table:
            all adjacent strike slopes
    """
    require_columns(
        surface,
        CONVEXITY_REQUIRED_COLUMNS,
        table_name="surface for butterfly convexity diagnostics",
    )

    if len(surface) == 0:
        raise ValueError("surface for butterfly convexity diagnostics is empty.")

    surface_id = str(surface["surface_id"].iloc[0])

    call_slopes = _build_adjacent_slope_table(
        surface,
        option_type="call",
        price_col="call_price",
    )

    put_slopes = _build_adjacent_slope_table(
        surface,
        option_type="put",
        price_col="put_price",
    )

    slope_table = pd.concat([call_slopes, put_slopes], ignore_index=True)

    call_triplets = _build_convexity_triplets_from_slopes(
        call_slopes,
        check_name="call_convex_in_strike",
        tolerance=tolerance,
    )

    put_triplets = _build_convexity_triplets_from_slopes(
        put_slopes,
        check_name="put_convex_in_strike",
        tolerance=tolerance,
    )

    triplet_table = pd.concat([call_triplets, put_triplets], ignore_index=True)

    if len(triplet_table) == 0:
        raise ValueError("No convexity triplets were generated. Check strike grid size.")

    failure_cols = [
        "surface_id",
        "diagnostic_group",
        "check_name",
        "option_type",
        "T",
        "lower_k_log",
        "middle_k_log",
        "upper_k_log",
        "lower_K",
        "middle_K",
        "upper_K",
        "lower_price",
        "middle_price",
        "upper_price",
        "slope_left",
        "slope_right",
        "slope_diff",
        "violation",
        "tolerance",
        "is_failure",
    ]

    failure_details = (
        triplet_table.loc[triplet_table["is_failure"], failure_cols]
        .sort_values("violation", ascending=False)
        .reset_index(drop=True)
    )

    records = []

    for check_name, option_type in [
        ("call_convex_in_strike", "call"),
        ("put_convex_in_strike", "put"),
    ]:
        subset = triplet_table.loc[
            (triplet_table["check_name"] == check_name)
            & (triplet_table["option_type"] == option_type)
        ].copy()

        failures = subset.loc[subset["is_failure"]].copy()

        failure_count = int(len(failures))
        max_violation = max_positive_or_zero(subset["violation"])

        if failure_count > 0:
            worst = failures.sort_values("violation", ascending=False).iloc[0]
        else:
            worst = pd.Series(dtype=object)

        if option_type == "call":
            pass_message = "All call price strike slopes are nondecreasing; call convexity passes."
            fail_message = "At least one call strike triplet violates convexity."
        else:
            pass_message = "All put price strike slopes are nondecreasing; put convexity passes."
            fail_message = "At least one put strike triplet violates convexity."

        records.append(
            make_diagnostic_record(
                surface_id=surface_id,
                diagnostic_group="butterfly_convexity",
                check_name=check_name,
                failure_count=failure_count,
                max_violation=max_violation,
                tolerance=tolerance,
                affected_T=worst.get("T", np.nan),
                affected_K=worst.get("middle_K", np.nan),
                affected_k_log=worst.get("middle_k_log", np.nan),
                affected_lower_K=worst.get("lower_K", np.nan),
                affected_middle_K=worst.get("middle_K", np.nan),
                affected_upper_K=worst.get("upper_K", np.nan),
                message=pass_message if failure_count == 0 else fail_message,
                clean_benchmark=clean_benchmark,
            )
        )

    ledger = diagnostic_records_to_frame(records)

    return ledger, failure_details, triplet_table, slope_table


# ------------------------------------------------------------
# Run butterfly convexity diagnostics on the clean positive-control surface
# ------------------------------------------------------------

(
    clean_convexity_ledger,
    clean_convexity_failures,
    clean_convexity_triplets,
    clean_convexity_slopes,
) = diagnose_butterfly_convexity(
    clean_surface,
    tolerance=TOL.convexity_abs,
    clean_benchmark=True,
)

clean_convexity_summary = summarize_diagnostic_ledger(clean_convexity_ledger)

clean_convexity_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,butterfly_convexity,call_convex_in_strike,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call price strike slopes are nondecreasing...
1,clean_bsm_constant_vol,butterfly_convexity,put_convex_in_strike,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put price strike slopes are nondecreasing;...


In [9]:
# ============================================================
# Cell 11: Calendar / total-variance term-structure diagnostics
# ============================================================

TERM_STRUCTURE_REQUIRED_COLUMNS = [
    "surface_id",
    "T",
    "k_log",
    "K",
    "total_variance_true",
]


def _build_total_variance_maturity_pairs(
    surface: pd.DataFrame,
    *,
    total_variance_col: str = "total_variance_true",
) -> pd.DataFrame:
    """
    Build adjacent maturity comparisons at fixed log-moneyness.

    For fixed log-moneyness k, total implied variance should be nondecreasing
    in maturity:

        T_1 < T_2  =>  w(k, T_1) <= w(k, T_2)

    This function compares adjacent maturities for each k_log value.

    The diagnostic is intentionally performed at fixed log-moneyness, not fixed
    raw strike, because the forward price changes with maturity.
    """
    require_columns(
        surface,
        ["surface_id", "T", "k_log", "K", total_variance_col],
        table_name="surface for total variance maturity-pair construction",
    )

    pair_rows = []

    for k_value, group in surface.groupby("k_log", sort=True):
        g = group.sort_values("T").reset_index(drop=True)

        if len(g) < 2:
            continue

        for i in range(len(g) - 1):
            near = g.iloc[i]
            far = g.iloc[i + 1]

            near_T = float(near["T"])
            far_T = float(far["T"])

            if far_T <= near_T:
                raise ValueError(
                    f"Non-increasing maturity grid detected at k_log={k_value}: "
                    f"near_T={near_T}, far_T={far_T}"
                )

            near_w = float(near[total_variance_col])
            far_w = float(far[total_variance_col])

            pair_rows.append(
                {
                    "surface_id": str(near["surface_id"]),
                    "diagnostic_group": "calendar_total_variance",
                    "check_name": "total_variance_nondecreasing_in_maturity",
                    "k_log": float(k_value),
                    "near_T": near_T,
                    "far_T": far_T,
                    "near_K": float(near["K"]),
                    "far_K": float(far["K"]),
                    "near_total_variance": near_w,
                    "far_total_variance": far_w,
                    "total_variance_diff_far_minus_near": float(far_w - near_w),
                }
            )

    return pd.DataFrame(pair_rows)


def diagnose_total_variance_term_structure(
    surface: pd.DataFrame,
    *,
    total_variance_col: str = "total_variance_true",
    tolerance: float = TOL.total_variance_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Diagnose total-variance monotonicity across maturities at fixed log-moneyness.

    For fixed k_log:

        w(k,T) = sigma_imp(k,T)^2 * T

    Calendar-spread consistency requires:

        w(k,T_{i+1}) - w(k,T_i) >= 0

    A failure occurs when:

        w(k,T_i) - w(k,T_{i+1}) > tolerance

    Returns:
        ledger:
            one standardized diagnostic record

        failure_details:
            failed adjacent maturity pairs

        pair_table:
            all adjacent maturity-pair comparisons
    """
    require_columns(
        surface,
        TERM_STRUCTURE_REQUIRED_COLUMNS,
        table_name="surface for calendar / total-variance diagnostics",
    )

    if total_variance_col not in surface.columns:
        raise ValueError(f"surface is missing total variance column: {total_variance_col}")

    if len(surface) == 0:
        raise ValueError("surface for calendar / total-variance diagnostics is empty.")

    surface_id = str(surface["surface_id"].iloc[0])

    pair_table = _build_total_variance_maturity_pairs(
        surface,
        total_variance_col=total_variance_col,
    )

    if len(pair_table) == 0:
        raise ValueError("No adjacent maturity pairs were generated. Check maturity grid size.")

    pair_table["violation"] = -pair_table["total_variance_diff_far_minus_near"]
    pair_table["tolerance"] = float(tolerance)
    pair_table["is_failure"] = pair_table["violation"] > float(tolerance)

    failure_cols = [
        "surface_id",
        "diagnostic_group",
        "check_name",
        "k_log",
        "near_T",
        "far_T",
        "near_K",
        "far_K",
        "near_total_variance",
        "far_total_variance",
        "total_variance_diff_far_minus_near",
        "violation",
        "tolerance",
        "is_failure",
    ]

    failure_details = (
        pair_table.loc[pair_table["is_failure"], failure_cols]
        .sort_values("violation", ascending=False)
        .reset_index(drop=True)
    )

    failure_count = int(len(failure_details))
    max_violation = max_positive_or_zero(pair_table["violation"])

    if failure_count > 0:
        worst = failure_details.iloc[0]
    else:
        worst = pd.Series(dtype=object)

    record = make_diagnostic_record(
        surface_id=surface_id,
        diagnostic_group="calendar_total_variance",
        check_name="total_variance_nondecreasing_in_maturity",
        failure_count=failure_count,
        max_violation=max_violation,
        tolerance=tolerance,
        affected_T=worst.get("far_T", np.nan),
        affected_K=worst.get("far_K", np.nan),
        affected_k_log=worst.get("k_log", np.nan),
        message=(
            "Total variance is nondecreasing across adjacent maturities at every fixed log-moneyness."
            if failure_count == 0
            else "At least one fixed-log-moneyness maturity pair has decreasing total variance."
        ),
        clean_benchmark=clean_benchmark,
    )

    ledger = diagnostic_records_to_frame([record])

    return ledger, failure_details, pair_table


# ------------------------------------------------------------
# Run total-variance term-structure diagnostics on the clean positive-control surface
# ------------------------------------------------------------

(
    clean_term_structure_ledger,
    clean_term_structure_failures,
    clean_term_structure_pairs,
) = diagnose_total_variance_term_structure(
    clean_surface,
    total_variance_col="total_variance_true",
    tolerance=TOL.total_variance_abs,
    clean_benchmark=True,
)

clean_term_structure_summary = summarize_diagnostic_ledger(clean_term_structure_ledger)

clean_term_structure_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,calendar_total_variance,total_variance_nondecreasing_in_maturity,PASS,0,0.0000000000,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,Total variance is nondecreasing across adjacen...


In [10]:
# ============================================================
# Cell 12: Implied-volatility inversion and IV consistency diagnostics
# ============================================================

IV_REQUIRED_COLUMNS = [
    "surface_id",
    "S0",
    "r",
    "q",
    "T",
    "k_log",
    "K",
    "call_price",
    "put_price",
    "sigma_true",
]


def bsm_vega(S0, K, T, sigma, r: float = r, q: float = q):
    """
    Black-Scholes-Merton vega with continuous dividend yield.

    Vega is the derivative of option value with respect to volatility:

        vega = S0 * exp(-qT) * phi(d1) * sqrt(T)

    This is the same for calls and puts under Black-Scholes-Merton.
    """
    scalar_output = _all_inputs_scalar(S0, K, T, sigma)

    S_arr, K_arr, T_arr, sig_arr = np.broadcast_arrays(
        _as_1d_float_array(S0),
        _as_1d_float_array(K),
        _as_1d_float_array(T),
        _as_1d_float_array(sigma),
    )

    if np.any(S_arr <= 0):
        raise ValueError("S0 must be strictly positive.")
    if np.any(K_arr <= 0):
        raise ValueError("K must be strictly positive.")
    if np.any(T_arr < 0):
        raise ValueError("T must be nonnegative.")
    if np.any(sig_arr < 0):
        raise ValueError("sigma must be nonnegative.")

    out = np.zeros_like(S_arr, dtype=float)

    active = (T_arr > 0.0) & (sig_arr > 0.0)

    if np.any(active):
        d1, _ = bsm_d1_d2(
            S0=S_arr[active],
            K=K_arr[active],
            T=T_arr[active],
            sigma=sig_arr[active],
            r=r,
            q=q,
        )

        out[active] = (
            S_arr[active]
            * np.exp(-q * T_arr[active])
            * norm.pdf(d1)
            * np.sqrt(T_arr[active])
        )

    return _maybe_scalar(out, scalar_output)


def _bsm_price_by_type(
    *,
    option_type: str,
    S0: float,
    K: float,
    T: float,
    sigma: float,
    r: float = r,
    q: float = q,
) -> float:
    """
    Dispatch Black-Scholes-Merton price by option type.
    """
    if option_type == "call":
        return float(
            bsm_call_price(
                S0=S0,
                K=K,
                T=T,
                sigma=sigma,
                r=r,
                q=q,
            )
        )

    if option_type == "put":
        return float(
            bsm_put_price(
                S0=S0,
                K=K,
                T=T,
                sigma=sigma,
                r=r,
                q=q,
            )
        )

    raise ValueError("option_type must be either 'call' or 'put'.")


def implied_vol_single(
    *,
    option_price: float,
    option_type: str,
    S0: float,
    K: float,
    T: float,
    r: float = r,
    q: float = q,
    vol_lower: float = 1.0e-12,
    vol_upper: float = 5.0,
    price_tolerance: float = TOL.price_abs,
) -> Dict[str, object]:
    """
    Invert one European option price into Black-Scholes implied volatility.

    The function returns a dictionary rather than only a float so that invalid
    cases can be diagnosed without hiding the reason.

    The inversion is bracketed with Brent's method. This is deliberate:
    the notebook is about diagnostic reliability, not speed.
    """
    option_price = float(option_price)
    S0 = float(S0)
    K = float(K)
    T = float(T)

    if option_type not in {"call", "put"}:
        raise ValueError("option_type must be either 'call' or 'put'.")

    if not np.isfinite(option_price):
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": "nonfinite_price",
            "pricing_error_at_iv": np.nan,
        }

    if S0 <= 0.0 or K <= 0.0 or T <= 0.0:
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": "invalid_contract_inputs",
            "pricing_error_at_iv": np.nan,
        }

    bounds = european_option_bounds(S0=S0, K=K, T=T, r=r, q=q).iloc[0]

    if option_type == "call":
        lower_bound = float(bounds["call_lower"])
        upper_bound = float(bounds["call_upper"])
    else:
        lower_bound = float(bounds["put_lower"])
        upper_bound = float(bounds["put_upper"])

    if option_price < lower_bound - price_tolerance:
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": "below_no_arbitrage_lower_bound",
            "pricing_error_at_iv": np.nan,
        }

    if option_price > upper_bound + price_tolerance:
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": "above_no_arbitrage_upper_bound",
            "pricing_error_at_iv": np.nan,
        }

    def objective(vol: float) -> float:
        return (
            _bsm_price_by_type(
                option_type=option_type,
                S0=S0,
                K=K,
                T=T,
                sigma=vol,
                r=r,
                q=q,
            )
            - option_price
        )

    f_low = objective(vol_lower)
    f_high = objective(vol_upper)

    # If the price is numerically indistinguishable from the near-zero-vol price,
    # return the lower bracket rather than forcing root finding into a flat region.
    if abs(f_low) <= price_tolerance:
        return {
            "iv": float(vol_lower),
            "iv_success": True,
            "iv_status": "near_lower_vol_boundary",
            "pricing_error_at_iv": float(f_low),
        }

    if abs(f_high) <= price_tolerance:
        return {
            "iv": float(vol_upper),
            "iv_success": True,
            "iv_status": "near_upper_vol_boundary",
            "pricing_error_at_iv": float(f_high),
        }

    if f_low * f_high > 0.0:
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": "root_not_bracketed",
            "pricing_error_at_iv": np.nan,
        }

    try:
        iv = brentq(
            objective,
            vol_lower,
            vol_upper,
            xtol=1.0e-12,
            rtol=1.0e-12,
            maxiter=200,
        )

        model_price = _bsm_price_by_type(
            option_type=option_type,
            S0=S0,
            K=K,
            T=T,
            sigma=iv,
            r=r,
            q=q,
        )

        return {
            "iv": float(iv),
            "iv_success": True,
            "iv_status": "converged",
            "pricing_error_at_iv": float(model_price - option_price),
        }

    except Exception as exc:
        return {
            "iv": np.nan,
            "iv_success": False,
            "iv_status": f"solver_error: {type(exc).__name__}",
            "pricing_error_at_iv": np.nan,
        }


def append_implied_vols_to_surface(
    surface: pd.DataFrame,
    *,
    price_tolerance: float = TOL.price_abs,
) -> pd.DataFrame:
    """
    Add call and put implied-volatility inversion results to a surface table.

    The output keeps both call-IV and put-IV results so that parity-consistent
    surfaces can be checked for call/put IV agreement.
    """
    require_columns(
        surface,
        IV_REQUIRED_COLUMNS,
        table_name="surface for implied-volatility inversion",
    )

    rows = []

    for row in surface.itertuples(index=False):
        row_dict = row._asdict()

        call_result = implied_vol_single(
            option_price=float(row_dict["call_price"]),
            option_type="call",
            S0=float(row_dict["S0"]),
            K=float(row_dict["K"]),
            T=float(row_dict["T"]),
            r=float(row_dict["r"]),
            q=float(row_dict["q"]),
            price_tolerance=price_tolerance,
        )

        put_result = implied_vol_single(
            option_price=float(row_dict["put_price"]),
            option_type="put",
            S0=float(row_dict["S0"]),
            K=float(row_dict["K"]),
            T=float(row_dict["T"]),
            r=float(row_dict["r"]),
            q=float(row_dict["q"]),
            price_tolerance=price_tolerance,
        )

        row_dict["call_iv"] = call_result["iv"]
        row_dict["call_iv_success"] = call_result["iv_success"]
        row_dict["call_iv_status"] = call_result["iv_status"]
        row_dict["call_iv_pricing_error"] = call_result["pricing_error_at_iv"]

        row_dict["put_iv"] = put_result["iv"]
        row_dict["put_iv_success"] = put_result["iv_success"]
        row_dict["put_iv_status"] = put_result["iv_status"]
        row_dict["put_iv_pricing_error"] = put_result["pricing_error_at_iv"]

        row_dict["call_total_variance_iv"] = (
            row_dict["call_iv"] ** 2 * row_dict["T"]
            if np.isfinite(row_dict["call_iv"])
            else np.nan
        )

        row_dict["put_total_variance_iv"] = (
            row_dict["put_iv"] ** 2 * row_dict["T"]
            if np.isfinite(row_dict["put_iv"])
            else np.nan
        )

        row_dict["call_vega_at_iv"] = (
            bsm_vega(
                S0=float(row_dict["S0"]),
                K=float(row_dict["K"]),
                T=float(row_dict["T"]),
                sigma=float(row_dict["call_iv"]),
                r=float(row_dict["r"]),
                q=float(row_dict["q"]),
            )
            if np.isfinite(row_dict["call_iv"])
            else np.nan
        )

        row_dict["put_vega_at_iv"] = (
            bsm_vega(
                S0=float(row_dict["S0"]),
                K=float(row_dict["K"]),
                T=float(row_dict["T"]),
                sigma=float(row_dict["put_iv"]),
                r=float(row_dict["r"]),
                q=float(row_dict["q"]),
            )
            if np.isfinite(row_dict["put_iv"])
            else np.nan
        )

        rows.append(row_dict)

    return pd.DataFrame(rows)


def diagnose_implied_vol_consistency(
    surface_with_iv: pd.DataFrame,
    *,
    iv_tolerance: float = 1.0e-8,
    price_tolerance: float = TOL.iv_abs,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Diagnose implied-volatility inversion consistency.

    Checks:

    1. all call IV inversions succeed;
    2. all put IV inversions succeed;
    3. call IV recovers the known synthetic volatility;
    4. put IV recovers the known synthetic volatility;
    5. call and put IV agree with each other;
    6. repriced call errors are small;
    7. repriced put errors are small.

    Price-level no-arbitrage remains primary. IV diagnostics are derived
    diagnostics built on top of admissible option prices.
    """
    required = [
        "surface_id",
        "T",
        "k_log",
        "K",
        "sigma_true",
        "call_iv",
        "put_iv",
        "call_iv_success",
        "put_iv_success",
        "call_iv_pricing_error",
        "put_iv_pricing_error",
    ]

    require_columns(
        surface_with_iv,
        required,
        table_name="surface with implied-volatility results",
    )

    if len(surface_with_iv) == 0:
        raise ValueError("surface with implied-volatility results is empty.")

    surface_id = str(surface_with_iv["surface_id"].iloc[0])
    work = surface_with_iv.copy()

    work["call_iv_error"] = work["call_iv"].astype(float) - work["sigma_true"].astype(float)
    work["put_iv_error"] = work["put_iv"].astype(float) - work["sigma_true"].astype(float)
    work["call_put_iv_difference"] = work["call_iv"].astype(float) - work["put_iv"].astype(float)

    check_specs = [
        {
            "check_name": "call_iv_inversion_success",
            "failure_mask": ~work["call_iv_success"].astype(bool),
            "violation_series": (~work["call_iv_success"].astype(bool)).astype(float),
            "tolerance": 0.0,
            "message_pass": "All call implied-volatility inversions succeeded.",
            "message_fail": "At least one call implied-volatility inversion failed.",
        },
        {
            "check_name": "put_iv_inversion_success",
            "failure_mask": ~work["put_iv_success"].astype(bool),
            "violation_series": (~work["put_iv_success"].astype(bool)).astype(float),
            "tolerance": 0.0,
            "message_pass": "All put implied-volatility inversions succeeded.",
            "message_fail": "At least one put implied-volatility inversion failed.",
        },
        {
            "check_name": "call_iv_recovers_true_sigma",
            "failure_mask": work["call_iv_error"].abs() > iv_tolerance,
            "violation_series": work["call_iv_error"].abs(),
            "tolerance": iv_tolerance,
            "message_pass": "Call IV recovers the synthetic generating volatility within tolerance.",
            "message_fail": "At least one call IV does not recover the synthetic generating volatility.",
        },
        {
            "check_name": "put_iv_recovers_true_sigma",
            "failure_mask": work["put_iv_error"].abs() > iv_tolerance,
            "violation_series": work["put_iv_error"].abs(),
            "tolerance": iv_tolerance,
            "message_pass": "Put IV recovers the synthetic generating volatility within tolerance.",
            "message_fail": "At least one put IV does not recover the synthetic generating volatility.",
        },
        {
            "check_name": "call_put_iv_agreement",
            "failure_mask": work["call_put_iv_difference"].abs() > iv_tolerance,
            "violation_series": work["call_put_iv_difference"].abs(),
            "tolerance": iv_tolerance,
            "message_pass": "Call IV and put IV agree within tolerance.",
            "message_fail": "At least one call-put pair has inconsistent implied volatilities.",
        },
        {
            "check_name": "call_iv_repricing_error",
            "failure_mask": work["call_iv_pricing_error"].abs() > price_tolerance,
            "violation_series": work["call_iv_pricing_error"].abs(),
            "tolerance": price_tolerance,
            "message_pass": "All call IV repricing errors are within tolerance.",
            "message_fail": "At least one call IV repricing error exceeds tolerance.",
        },
        {
            "check_name": "put_iv_repricing_error",
            "failure_mask": work["put_iv_pricing_error"].abs() > price_tolerance,
            "violation_series": work["put_iv_pricing_error"].abs(),
            "tolerance": price_tolerance,
            "message_pass": "All put IV repricing errors are within tolerance.",
            "message_fail": "At least one put IV repricing error exceeds tolerance.",
        },
    ]

    records = []
    failure_tables = []

    for spec in check_specs:
        tmp = work.copy()
        tmp["diagnostic_group"] = "implied_volatility_consistency"
        tmp["check_name"] = spec["check_name"]
        tmp["violation"] = np.asarray(spec["violation_series"], dtype=float)
        tmp["tolerance"] = float(spec["tolerance"])
        tmp["is_failure"] = np.asarray(spec["failure_mask"], dtype=bool)

        failures = tmp.loc[tmp["is_failure"]].copy()

        failure_cols = [
            "surface_id",
            "diagnostic_group",
            "check_name",
            "T",
            "k_log",
            "K",
            "sigma_true",
            "call_iv",
            "put_iv",
            "call_iv_status",
            "put_iv_status",
            "call_iv_pricing_error",
            "put_iv_pricing_error",
            "violation",
            "tolerance",
            "is_failure",
        ]

        failure_tables.append(failures[failure_cols])

        failure_count = int(len(failures))
        max_violation = max_abs_or_zero(tmp["violation"])

        if failure_count > 0:
            worst = failures.sort_values("violation", ascending=False).iloc[0]
        else:
            worst = pd.Series(dtype=object)

        records.append(
            make_diagnostic_record(
                surface_id=surface_id,
                diagnostic_group="implied_volatility_consistency",
                check_name=spec["check_name"],
                failure_count=failure_count,
                max_violation=max_violation,
                tolerance=float(spec["tolerance"]),
                affected_T=worst.get("T", np.nan),
                affected_K=worst.get("K", np.nan),
                affected_k_log=worst.get("k_log", np.nan),
                message=spec["message_pass"] if failure_count == 0 else spec["message_fail"],
                clean_benchmark=clean_benchmark,
            )
        )

    ledger = diagnostic_records_to_frame(records)

    failure_details = (
        pd.concat(failure_tables, ignore_index=True)
        if len(failure_tables) > 0
        else pd.DataFrame()
    )

    return ledger, failure_details


# ------------------------------------------------------------
# Run IV inversion and IV consistency diagnostics on the clean positive-control surface
# ------------------------------------------------------------

clean_surface_iv = append_implied_vols_to_surface(
    clean_surface,
    price_tolerance=TOL.price_abs,
)

clean_iv_ledger, clean_iv_failures = diagnose_implied_vol_consistency(
    clean_surface_iv,
    iv_tolerance=1.0e-8,
    price_tolerance=TOL.iv_abs,
    clean_benchmark=True,
)

clean_iv_summary = summarize_diagnostic_ledger(clean_iv_ledger)

clean_iv_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_inversion_success,PASS,0,0.0000000000,0.0000000000,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call implied-volatility inversions succeeded.
1,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_inversion_success,PASS,0,0.0000000000,0.0000000000,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put implied-volatility inversions succeeded.
2,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_recovers_true_sigma,FAIL,1,0.2000000000,0.0000000100,CRITICAL,0.1000000000,67.1662027662,-0.4000000000,NaN,NaN,NaN,At least one call IV does not recover the synt...
3,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_recovers_true_sigma,FAIL,2,0.2000000000,0.0000000100,CRITICAL,0.1000000000,67.1662027662,-0.4000000000,NaN,NaN,NaN,At least one put IV does not recover the synth...
4,clean_bsm_constant_vol,implied_volatility_consistency,call_put_iv_agreement,FAIL,1,0.0000003314,0.0000000100,CRITICAL,0.1000000000,149.4811332676,0.4000000000,NaN,NaN,NaN,At least one call-put pair has inconsistent im...
5,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_repricing_error,PASS,0,0.0000000001,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call IV repricing errors are within tolera...
6,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_repricing_error,PASS,0,0.0000000001,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put IV repricing errors are within tolerance.


In [11]:
# ============================================================
# Cell 13: Low-vega-aware implied-volatility diagnostics
# ============================================================

IV_VEGA_FLOOR = 1.0e-7
IV_RECOVERY_TOL = 1.0e-8
IV_AGREEMENT_TOL = 1.0e-8
IV_REPRICE_TOL = TOL.iv_abs


def append_iv_identifiability_flags(
    surface_with_iv: pd.DataFrame,
    *,
    vega_floor: float = IV_VEGA_FLOOR,
) -> pd.DataFrame:
    """
    Add implied-volatility identifiability flags.

    IV inversion can be numerically successful but economically uninformative
    when option value is extremely close to its no-arbitrage lower bound.

    In those cases, many volatility values can reprice the option almost equally
    well because vega is nearly zero. The correct diagnostic treatment is not:

        "IV failed"

    but rather:

        "IV is not identifiable from price at the requested tolerance."

    This distinction matters for deep OTM / deep ITM options, especially at
    short maturity.
    """
    required = [
        "call_iv",
        "put_iv",
        "call_iv_success",
        "put_iv_success",
        "call_iv_status",
        "put_iv_status",
        "call_vega_at_iv",
        "put_vega_at_iv",
        "call_price",
        "put_price",
        "call_lower",
        "put_lower",
        "T",
        "sigma_true",
    ]

    require_columns(
        surface_with_iv,
        required,
        table_name="surface with implied-volatility results",
    )

    out = surface_with_iv.copy()

    out["call_time_value"] = out["call_price"].astype(float) - out["call_lower"].astype(float)
    out["put_time_value"] = out["put_price"].astype(float) - out["put_lower"].astype(float)

    out["call_iv_identifiable"] = (
        out["call_iv_success"].astype(bool)
        & np.isfinite(out["call_iv"].astype(float))
        & (out["call_iv_status"].astype(str) == "converged")
        & (out["call_vega_at_iv"].astype(float) > float(vega_floor))
    )

    out["put_iv_identifiable"] = (
        out["put_iv_success"].astype(bool)
        & np.isfinite(out["put_iv"].astype(float))
        & (out["put_iv_status"].astype(str) == "converged")
        & (out["put_vega_at_iv"].astype(float) > float(vega_floor))
    )

    out["call_iv_low_information"] = (
        out["call_iv_success"].astype(bool)
        & ~out["call_iv_identifiable"].astype(bool)
    )

    out["put_iv_low_information"] = (
        out["put_iv_success"].astype(bool)
        & ~out["put_iv_identifiable"].astype(bool)
    )

    out["call_iv_error"] = out["call_iv"].astype(float) - out["sigma_true"].astype(float)
    out["put_iv_error"] = out["put_iv"].astype(float) - out["sigma_true"].astype(float)
    out["call_put_iv_difference"] = out["call_iv"].astype(float) - out["put_iv"].astype(float)

    out["both_call_put_iv_identifiable"] = (
        out["call_iv_identifiable"].astype(bool)
        & out["put_iv_identifiable"].astype(bool)
    )

    return out


def diagnose_implied_vol_consistency_low_vega_aware(
    surface_with_iv: pd.DataFrame,
    *,
    iv_recovery_tolerance: float = IV_RECOVERY_TOL,
    iv_agreement_tolerance: float = IV_AGREEMENT_TOL,
    price_tolerance: float = IV_REPRICE_TOL,
    vega_floor: float = IV_VEGA_FLOOR,
    clean_benchmark: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Diagnose implied-volatility consistency with low-vega awareness.

    The diagnostic separates:

    1. inversion success;
    2. identifiability;
    3. recovery of known synthetic volatility on identifiable rows;
    4. call-put IV agreement on jointly identifiable rows;
    5. repricing error.

    This avoids falsely treating low-vega near-intrinsic options as model
    failures. If an option has almost zero vega, implied volatility is not a
    stable object even when the price is admissible.
    """
    work = append_iv_identifiability_flags(
        surface_with_iv,
        vega_floor=vega_floor,
    )

    surface_id = str(work["surface_id"].iloc[0])

    call_success_fail = ~work["call_iv_success"].astype(bool)
    put_success_fail = ~work["put_iv_success"].astype(bool)

    call_ident = work["call_iv_identifiable"].astype(bool)
    put_ident = work["put_iv_identifiable"].astype(bool)
    both_ident = work["both_call_put_iv_identifiable"].astype(bool)

    call_recovery_fail = pd.Series(False, index=work.index)
    put_recovery_fail = pd.Series(False, index=work.index)
    call_put_agreement_fail = pd.Series(False, index=work.index)

    call_recovery_fail.loc[call_ident] = (
        work.loc[call_ident, "call_iv_error"].abs() > iv_recovery_tolerance
    )

    put_recovery_fail.loc[put_ident] = (
        work.loc[put_ident, "put_iv_error"].abs() > iv_recovery_tolerance
    )

    call_put_agreement_fail.loc[both_ident] = (
        work.loc[both_ident, "call_put_iv_difference"].abs() > iv_agreement_tolerance
    )

    call_reprice_fail = work["call_iv_pricing_error"].abs() > price_tolerance
    put_reprice_fail = work["put_iv_pricing_error"].abs() > price_tolerance

    check_specs = [
        {
            "check_name": "call_iv_inversion_success",
            "failure_mask": call_success_fail,
            "violation_series": call_success_fail.astype(float),
            "tolerance": 0.0,
            "message_pass": "All call implied-volatility inversions returned admissible diagnostic results.",
            "message_fail": "At least one call implied-volatility inversion failed.",
        },
        {
            "check_name": "put_iv_inversion_success",
            "failure_mask": put_success_fail,
            "violation_series": put_success_fail.astype(float),
            "tolerance": 0.0,
            "message_pass": "All put implied-volatility inversions returned admissible diagnostic results.",
            "message_fail": "At least one put implied-volatility inversion failed.",
        },
        {
            "check_name": "call_iv_recovers_true_sigma_when_identifiable",
            "failure_mask": call_recovery_fail,
            "violation_series": np.where(
                call_ident,
                work["call_iv_error"].abs(),
                0.0,
            ),
            "tolerance": iv_recovery_tolerance,
            "message_pass": "Call IV recovers true sigma on identifiable rows; low-vega rows are not treated as recovery failures.",
            "message_fail": "At least one identifiable call IV does not recover the synthetic generating volatility.",
        },
        {
            "check_name": "put_iv_recovers_true_sigma_when_identifiable",
            "failure_mask": put_recovery_fail,
            "violation_series": np.where(
                put_ident,
                work["put_iv_error"].abs(),
                0.0,
            ),
            "tolerance": iv_recovery_tolerance,
            "message_pass": "Put IV recovers true sigma on identifiable rows; low-vega rows are not treated as recovery failures.",
            "message_fail": "At least one identifiable put IV does not recover the synthetic generating volatility.",
        },
        {
            "check_name": "call_put_iv_agreement_when_identifiable",
            "failure_mask": call_put_agreement_fail,
            "violation_series": np.where(
                both_ident,
                work["call_put_iv_difference"].abs(),
                0.0,
            ),
            "tolerance": iv_agreement_tolerance,
            "message_pass": "Call and put IV agree on jointly identifiable rows.",
            "message_fail": "At least one jointly identifiable call-put pair has inconsistent implied volatilities.",
        },
        {
            "check_name": "call_iv_repricing_error",
            "failure_mask": call_reprice_fail,
            "violation_series": work["call_iv_pricing_error"].abs(),
            "tolerance": price_tolerance,
            "message_pass": "All call IV repricing errors are within tolerance.",
            "message_fail": "At least one call IV repricing error exceeds tolerance.",
        },
        {
            "check_name": "put_iv_repricing_error",
            "failure_mask": put_reprice_fail,
            "violation_series": work["put_iv_pricing_error"].abs(),
            "tolerance": price_tolerance,
            "message_pass": "All put IV repricing errors are within tolerance.",
            "message_fail": "At least one put IV repricing error exceeds tolerance.",
        },
    ]

    records = []
    failure_tables = []

    for spec in check_specs:
        tmp = work.copy()
        tmp["diagnostic_group"] = "implied_volatility_consistency"
        tmp["check_name"] = spec["check_name"]
        tmp["violation"] = np.asarray(spec["violation_series"], dtype=float)
        tmp["tolerance"] = float(spec["tolerance"])
        tmp["is_failure"] = np.asarray(spec["failure_mask"], dtype=bool)

        failures = tmp.loc[tmp["is_failure"]].copy()

        failure_cols = [
            "surface_id",
            "diagnostic_group",
            "check_name",
            "T",
            "k_log",
            "K",
            "sigma_true",
            "call_iv",
            "put_iv",
            "call_iv_status",
            "put_iv_status",
            "call_iv_identifiable",
            "put_iv_identifiable",
            "call_vega_at_iv",
            "put_vega_at_iv",
            "call_iv_pricing_error",
            "put_iv_pricing_error",
            "violation",
            "tolerance",
            "is_failure",
        ]

        failure_tables.append(failures[failure_cols])

        failure_count = int(len(failures))
        max_violation = max_abs_or_zero(tmp["violation"])

        if failure_count > 0:
            worst = failures.sort_values("violation", ascending=False).iloc[0]
        else:
            worst = pd.Series(dtype=object)

        records.append(
            make_diagnostic_record(
                surface_id=surface_id,
                diagnostic_group="implied_volatility_consistency",
                check_name=spec["check_name"],
                failure_count=failure_count,
                max_violation=max_violation,
                tolerance=float(spec["tolerance"]),
                affected_T=worst.get("T", np.nan),
                affected_K=worst.get("K", np.nan),
                affected_k_log=worst.get("k_log", np.nan),
                message=spec["message_pass"] if failure_count == 0 else spec["message_fail"],
                clean_benchmark=clean_benchmark,
            )
        )

    ledger = diagnostic_records_to_frame(records)

    failure_details = (
        pd.concat(failure_tables, ignore_index=True)
        if len(failure_tables) > 0
        else pd.DataFrame()
    )

    identifiability_summary = pd.DataFrame(
        {
            "metric": [
                "rows",
                "call_iv_success_count",
                "put_iv_success_count",
                "call_iv_identifiable_count",
                "put_iv_identifiable_count",
                "jointly_identifiable_count",
                "call_low_information_count",
                "put_low_information_count",
                "min_call_vega_at_iv",
                "min_put_vega_at_iv",
                "max_abs_call_iv_error_identifiable",
                "max_abs_put_iv_error_identifiable",
                "max_abs_call_put_iv_difference_jointly_identifiable",
            ],
            "value": [
                len(work),
                int(work["call_iv_success"].sum()),
                int(work["put_iv_success"].sum()),
                int(work["call_iv_identifiable"].sum()),
                int(work["put_iv_identifiable"].sum()),
                int(work["both_call_put_iv_identifiable"].sum()),
                int(work["call_iv_low_information"].sum()),
                int(work["put_iv_low_information"].sum()),
                float(np.nanmin(work["call_vega_at_iv"])),
                float(np.nanmin(work["put_vega_at_iv"])),
                max_abs_or_zero(work.loc[call_ident, "call_iv_error"]),
                max_abs_or_zero(work.loc[put_ident, "put_iv_error"]),
                max_abs_or_zero(work.loc[both_ident, "call_put_iv_difference"]),
            ],
        }
    )

    return ledger, failure_details, identifiability_summary


# ------------------------------------------------------------
# Rerun IV diagnostics with low-vega identifiability separation
# ------------------------------------------------------------

(
    clean_iv_ledger,
    clean_iv_failures,
    clean_iv_identifiability_summary,
) = diagnose_implied_vol_consistency_low_vega_aware(
    clean_surface_iv,
    iv_recovery_tolerance=IV_RECOVERY_TOL,
    iv_agreement_tolerance=IV_AGREEMENT_TOL,
    price_tolerance=IV_REPRICE_TOL,
    vega_floor=IV_VEGA_FLOOR,
    clean_benchmark=True,
)

clean_iv_summary = summarize_diagnostic_ledger(clean_iv_ledger)

clean_iv_ledger

,surface_id,diagnostic_group,check_name,status,failure_count,max_violation,tolerance,severity,affected_T,affected_K,affected_k_log,affected_lower_K,affected_middle_K,affected_upper_K,message
0,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_inversion_success,PASS,0,0.0000000000,0.0000000000,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call implied-volatility inversions returne...
1,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_inversion_success,PASS,0,0.0000000000,0.0000000000,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put implied-volatility inversions returned...
2,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_recovers_true_sigma_when_identifiable,PASS,0,0.0000000001,0.0000000100,INFO,NaN,NaN,NaN,NaN,NaN,NaN,Call IV recovers true sigma on identifiable ro...
3,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_recovers_true_sigma_when_identifiable,PASS,0,0.0000000001,0.0000000100,INFO,NaN,NaN,NaN,NaN,NaN,NaN,Put IV recovers true sigma on identifiable row...
4,clean_bsm_constant_vol,implied_volatility_consistency,call_put_iv_agreement_when_identifiable,PASS,0,0.0000000001,0.0000000100,INFO,NaN,NaN,NaN,NaN,NaN,NaN,Call and put IV agree on jointly identifiable ...
5,clean_bsm_constant_vol,implied_volatility_consistency,call_iv_repricing_error,PASS,0,0.0000000001,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All call IV repricing errors are within tolera...
6,clean_bsm_constant_vol,implied_volatility_consistency,put_iv_repricing_error,PASS,0,0.0000000001,0.0000000001,INFO,NaN,NaN,NaN,NaN,NaN,NaN,All put IV repricing errors are within tolerance.


In [12]:
# ============================================================
# Cell 14: Clean positive-control diagnostic ledger
# ============================================================

def combine_diagnostic_ledgers(*ledgers: pd.DataFrame) -> pd.DataFrame:
    """
    Combine multiple diagnostic ledgers into one stable notebook-level ledger.
    """
    valid_ledgers = []

    for ledger in ledgers:
        if ledger is None or len(ledger) == 0:
            continue

        require_columns(
            ledger,
            DIAGNOSTIC_RECORD_COLUMNS,
            table_name="diagnostic ledger to combine",
        )

        valid_ledgers.append(ledger.copy())

    if len(valid_ledgers) == 0:
        return pd.DataFrame(columns=DIAGNOSTIC_RECORD_COLUMNS)

    combined = pd.concat(valid_ledgers, ignore_index=True)

    combined["status"] = combined["status"].astype(str)
    combined["severity"] = combined["severity"].astype(str)
    combined["failure_count"] = combined["failure_count"].astype(int)
    combined["max_violation"] = combined["max_violation"].astype(float)
    combined["tolerance"] = combined["tolerance"].astype(float)

    return combined[DIAGNOSTIC_RECORD_COLUMNS].copy()


def clean_positive_control_readiness_report(ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Build a compact readiness report for the clean benchmark surface.

    For the positive control, any failed diagnostic is serious because the
    surface was generated from internally consistent Black-Scholes prices.
    """
    require_columns(
        ledger,
        [
            "diagnostic_group",
            "check_name",
            "status",
            "failure_count",
            "max_violation",
            "severity",
        ],
        table_name="clean positive-control ledger",
    )

    total_checks = int(len(ledger))
    failed_checks = int((ledger["status"] == STATUS_FAIL).sum())
    total_failures = int(ledger["failure_count"].sum())
    critical_checks = int((ledger["severity"] == SEVERITY_CRITICAL).sum())

    ready = (
        total_checks > 0
        and failed_checks == 0
        and total_failures == 0
        and critical_checks == 0
    )

    report = pd.DataFrame(
        {
            "metric": [
                "surface_id",
                "total_diagnostic_checks",
                "failed_diagnostic_checks",
                "total_failure_count",
                "critical_severity_checks",
                "max_recorded_violation",
                "clean_positive_control_ready",
            ],
            "value": [
                str(ledger["surface_id"].iloc[0]) if total_checks > 0 else "",
                total_checks,
                failed_checks,
                total_failures,
                critical_checks,
                float(ledger["max_violation"].max()) if total_checks > 0 else np.nan,
                bool(ready),
            ],
        }
    )

    return report


clean_positive_control_ledger = combine_diagnostic_ledgers(
    clean_pointwise_ledger,
    clean_parity_ledger,
    clean_vertical_ledger,
    clean_convexity_ledger,
    clean_term_structure_ledger,
    clean_iv_ledger,
)

clean_positive_control_summary = summarize_diagnostic_ledger(
    clean_positive_control_ledger
)

clean_positive_control_report = clean_positive_control_readiness_report(
    clean_positive_control_ledger
)


# ------------------------------------------------------------
# Failure detail audit
# ------------------------------------------------------------

clean_failure_detail_counts = pd.DataFrame(
    {
        "failure_table": [
            "clean_pointwise_failures",
            "clean_parity_failures",
            "clean_vertical_failures",
            "clean_convexity_failures",
            "clean_term_structure_failures",
            "clean_iv_failures",
        ],
        "failure_rows": [
            len(clean_pointwise_failures),
            len(clean_parity_failures),
            len(clean_vertical_failures),
            len(clean_convexity_failures),
            len(clean_term_structure_failures),
            len(clean_iv_failures),
        ],
    }
)

clean_failure_detail_counts["status"] = np.where(
    clean_failure_detail_counts["failure_rows"] == 0,
    STATUS_PASS,
    STATUS_FAIL,
)


# ------------------------------------------------------------
# Hard assertion for the positive control
# ------------------------------------------------------------

if not bool(clean_positive_control_report.loc[
    clean_positive_control_report["metric"] == "clean_positive_control_ready",
    "value",
].iloc[0]):
    display(clean_positive_control_ledger)
    display(clean_positive_control_summary)
    display(clean_failure_detail_counts)
    raise AssertionError(
        "Clean Black-Scholes positive-control surface failed at least one diagnostic."
    )


clean_positive_control_report

,metric,value
0,surface_id,clean_bsm_constant_vol
1,total_diagnostic_checks,17
2,failed_diagnostic_checks,0
3,total_failure_count,0
4,critical_severity_checks,0
5,max_recorded_violation,0.0000000001
6,clean_positive_control_ready,True


In [13]:
# ============================================================
# Cell 15: Negative-control surface builders and diagnostic runner
# ============================================================

def refresh_surface_derived_columns(surface: pd.DataFrame) -> pd.DataFrame:
    """
    Refresh derived columns after deliberate corruption.

    The no-arbitrage bounds are not changed by price corruption because they
    depend only on S0, K, T, r, q, F, and D.

    This function refreshes:
    - parity residual;
    - call / put time value;
    - bound slack columns.

    It does not repair the surface.
    """
    required = [
        "S0",
        "r",
        "q",
        "T",
        "K",
        "F",
        "D",
        "call_price",
        "put_price",
        "call_lower",
        "call_upper",
        "put_lower",
        "put_upper",
    ]

    require_columns(
        surface,
        required,
        table_name="surface to refresh derived columns",
    )

    out = surface.copy()

    out["parity_residual"] = out.apply(
        lambda row: put_call_parity_residual(
            call_price=float(row["call_price"]),
            put_price=float(row["put_price"]),
            S0=float(row["S0"]),
            K=float(row["K"]),
            T=float(row["T"]),
            r=float(row["r"]),
            q=float(row["q"]),
        ),
        axis=1,
    )

    out["call_time_value"] = out["call_price"].astype(float) - out["call_lower"].astype(float)
    out["put_time_value"] = out["put_price"].astype(float) - out["put_lower"].astype(float)

    out["call_bound_lower_slack"] = out["call_price"].astype(float) - out["call_lower"].astype(float)
    out["call_bound_upper_slack"] = out["call_upper"].astype(float) - out["call_price"].astype(float)
    out["put_bound_lower_slack"] = out["put_price"].astype(float) - out["put_lower"].astype(float)
    out["put_bound_upper_slack"] = out["put_upper"].astype(float) - out["put_price"].astype(float)

    return out


def copy_surface_with_id(surface: pd.DataFrame, surface_id: str) -> pd.DataFrame:
    """
    Copy a surface and assign a new surface_id.
    """
    out = surface.copy(deep=True)
    out["surface_id"] = str(surface_id)
    return refresh_surface_derived_columns(out)


def _select_single_surface_row(
    surface: pd.DataFrame,
    *,
    T_value: float,
    k_log_value: float,
) -> int:
    """
    Select a single row by exact synthetic maturity and log-moneyness value.

    The synthetic grid is deterministic, so exact matching is acceptable after
    using np.isclose.
    """
    require_columns(
        surface,
        ["T", "k_log"],
        table_name="surface for row selection",
    )

    mask = (
        np.isclose(surface["T"].astype(float), float(T_value))
        & np.isclose(surface["k_log"].astype(float), float(k_log_value))
    )

    matches = surface.loc[mask]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one row for T={T_value}, k_log={k_log_value}; "
            f"found {len(matches)}."
        )

    return int(matches.index[0])


def corrupt_call_below_lower_bound(
    surface: pd.DataFrame,
    *,
    T_value: float = 0.25,
    k_log_value: float = -0.20,
    breach: float = 0.05,
    surface_id: str = "negative_call_below_lower_bound",
) -> pd.DataFrame:
    """
    Negative control: force one call price below its lower no-arbitrage bound.
    """
    out = copy_surface_with_id(surface, surface_id)
    idx = _select_single_surface_row(out, T_value=T_value, k_log_value=k_log_value)

    out.loc[idx, "call_price"] = float(out.loc[idx, "call_lower"]) - float(breach)

    return refresh_surface_derived_columns(out)


def corrupt_put_above_upper_bound(
    surface: pd.DataFrame,
    *,
    T_value: float = 0.25,
    k_log_value: float = 0.20,
    breach: float = 0.05,
    surface_id: str = "negative_put_above_upper_bound",
) -> pd.DataFrame:
    """
    Negative control: force one put price above its upper no-arbitrage bound.
    """
    out = copy_surface_with_id(surface, surface_id)
    idx = _select_single_surface_row(out, T_value=T_value, k_log_value=k_log_value)

    out.loc[idx, "put_price"] = float(out.loc[idx, "put_upper"]) + float(breach)

    return refresh_surface_derived_columns(out)


def corrupt_call_vertical_monotonicity(
    surface: pd.DataFrame,
    *,
    T_value: float = 0.50,
    lower_k_log: float = 0.00,
    higher_k_log: float = 0.05,
    breach: float = 0.10,
    surface_id: str = "negative_call_vertical_violation",
) -> pd.DataFrame:
    """
    Negative control: make a higher-strike call more expensive than the adjacent
    lower-strike call.

    This creates a vertical-spread monotonicity violation.
    """
    out = copy_surface_with_id(surface, surface_id)

    lower_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=lower_k_log,
    )

    higher_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=higher_k_log,
    )

    lower_price = float(out.loc[lower_idx, "call_price"])
    out.loc[higher_idx, "call_price"] = lower_price + float(breach)

    return refresh_surface_derived_columns(out)


def corrupt_put_vertical_monotonicity(
    surface: pd.DataFrame,
    *,
    T_value: float = 0.50,
    lower_k_log: float = -0.05,
    higher_k_log: float = 0.00,
    breach: float = 0.10,
    surface_id: str = "negative_put_vertical_violation",
) -> pd.DataFrame:
    """
    Negative control: make a higher-strike put cheaper than the adjacent
    lower-strike put.

    This creates a vertical-spread monotonicity violation.
    """
    out = copy_surface_with_id(surface, surface_id)

    lower_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=lower_k_log,
    )

    higher_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=higher_k_log,
    )

    lower_price = float(out.loc[lower_idx, "put_price"])
    out.loc[higher_idx, "put_price"] = max(lower_price - float(breach), 0.0)

    return refresh_surface_derived_columns(out)


def corrupt_call_butterfly_convexity(
    surface: pd.DataFrame,
    *,
    T_value: float = 1.00,
    lower_k_log: float = -0.05,
    middle_k_log: float = 0.00,
    upper_k_log: float = 0.05,
    breach: float = 0.50,
    surface_id: str = "negative_call_butterfly_violation",
) -> pd.DataFrame:
    """
    Negative control: inflate the middle call price enough to break convexity.

    For a convex price curve, the middle point should not sit too high relative
    to the adjacent strike slopes. Raising the middle point creates a local
    concavity / butterfly violation.
    """
    out = copy_surface_with_id(surface, surface_id)

    lower_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=lower_k_log,
    )

    middle_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=middle_k_log,
    )

    upper_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=upper_k_log,
    )

    lower_price = float(out.loc[lower_idx, "call_price"])
    upper_price = float(out.loc[upper_idx, "call_price"])

    out.loc[middle_idx, "call_price"] = max(lower_price, upper_price) + float(breach)

    return refresh_surface_derived_columns(out)


def corrupt_put_butterfly_convexity(
    surface: pd.DataFrame,
    *,
    T_value: float = 1.00,
    lower_k_log: float = -0.05,
    middle_k_log: float = 0.00,
    upper_k_log: float = 0.05,
    breach: float = 0.50,
    surface_id: str = "negative_put_butterfly_violation",
) -> pd.DataFrame:
    """
    Negative control: inflate the middle put price enough to break convexity.

    This creates a local butterfly violation in the put curve.
    """
    out = copy_surface_with_id(surface, surface_id)

    lower_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=lower_k_log,
    )

    middle_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=middle_k_log,
    )

    upper_idx = _select_single_surface_row(
        out,
        T_value=T_value,
        k_log_value=upper_k_log,
    )

    lower_price = float(out.loc[lower_idx, "put_price"])
    upper_price = float(out.loc[upper_idx, "put_price"])

    out.loc[middle_idx, "put_price"] = max(lower_price, upper_price) + float(breach)

    return refresh_surface_derived_columns(out)


def corrupt_put_call_parity_only(
    surface: pd.DataFrame,
    *,
    T_value: float = 0.50,
    k_log_value: float = 0.00,
    call_bump: float = 0.25,
    surface_id: str = "negative_put_call_parity_violation",
) -> pd.DataFrame:
    """
    Negative control: bump one call price while leaving the matching put
    unchanged.

    The bump is chosen to be small enough that pointwise bounds typically still
    pass, but parity should fail.
    """
    out = copy_surface_with_id(surface, surface_id)
    idx = _select_single_surface_row(out, T_value=T_value, k_log_value=k_log_value)

    proposed_call = float(out.loc[idx, "call_price"]) + float(call_bump)
    call_upper = float(out.loc[idx, "call_upper"])

    if proposed_call >= call_upper:
        raise ValueError(
            "call_bump is too large and would also create a call upper-bound violation."
        )

    out.loc[idx, "call_price"] = proposed_call

    return refresh_surface_derived_columns(out)


def corrupt_total_variance_term_structure(
    surface: pd.DataFrame,
    *,
    k_log_value: float = 0.00,
    near_T: float = 0.50,
    far_T: float = 1.00,
    far_total_variance: float = 0.005,
    surface_id: str = "negative_total_variance_calendar_violation",
) -> pd.DataFrame:
    """
    Negative control: force total variance to decrease with maturity at fixed
    log-moneyness.

    This corrupts the total-variance column used by the diagnostic. It does not
    reprice the options, because the goal is to test the term-structure
    diagnostic directly.
    """
    out = copy_surface_with_id(surface, surface_id)

    near_idx = _select_single_surface_row(out, T_value=near_T, k_log_value=k_log_value)
    far_idx = _select_single_surface_row(out, T_value=far_T, k_log_value=k_log_value)

    near_w = float(out.loc[near_idx, "total_variance_true"])

    if far_total_variance >= near_w:
        raise ValueError(
            "far_total_variance must be lower than the near maturity total variance "
            "to create a calendar / term-structure violation."
        )

    out.loc[far_idx, "total_variance_true"] = float(far_total_variance)

    return refresh_surface_derived_columns(out)


def run_static_arbitrage_diagnostic_suite(
    surface: pd.DataFrame,
    *,
    clean_benchmark: bool = False,
    run_iv_diagnostics: bool = True,
) -> Dict[str, object]:
    """
    Run the full static-arbitrage diagnostic suite on one surface.

    This function is used for negative controls so every deliberately corrupted
    surface is passed through the same diagnostic machinery.

    Returns a dictionary containing ledgers, summaries, failure details, and
    intermediate tables.
    """
    pointwise_ledger, pointwise_failures = diagnose_pointwise_bounds(
        surface,
        tolerance=TOL.price_abs,
        clean_benchmark=clean_benchmark,
    )

    parity_ledger, parity_failures = diagnose_put_call_parity(
        surface,
        tolerance=TOL.parity_abs,
        clean_benchmark=clean_benchmark,
    )

    vertical_ledger, vertical_failures, vertical_pairs = diagnose_vertical_monotonicity(
        surface,
        tolerance=TOL.monotonicity_abs,
        clean_benchmark=clean_benchmark,
    )

    (
        convexity_ledger,
        convexity_failures,
        convexity_triplets,
        convexity_slopes,
    ) = diagnose_butterfly_convexity(
        surface,
        tolerance=TOL.convexity_abs,
        clean_benchmark=clean_benchmark,
    )

    term_structure_ledger, term_structure_failures, term_structure_pairs = diagnose_total_variance_term_structure(
        surface,
        total_variance_col="total_variance_true",
        tolerance=TOL.total_variance_abs,
        clean_benchmark=clean_benchmark,
    )

    ledgers = [
        pointwise_ledger,
        parity_ledger,
        vertical_ledger,
        convexity_ledger,
        term_structure_ledger,
    ]

    iv_surface = None
    iv_ledger = diagnostic_records_to_frame([])
    iv_failures = pd.DataFrame()
    iv_identifiability_summary = pd.DataFrame()

    if run_iv_diagnostics:
        iv_surface = append_implied_vols_to_surface(
            surface,
            price_tolerance=TOL.price_abs,
        )

        iv_ledger, iv_failures, iv_identifiability_summary = diagnose_implied_vol_consistency_low_vega_aware(
            iv_surface,
            iv_recovery_tolerance=IV_RECOVERY_TOL,
            iv_agreement_tolerance=IV_AGREEMENT_TOL,
            price_tolerance=IV_REPRICE_TOL,
            vega_floor=IV_VEGA_FLOOR,
            clean_benchmark=clean_benchmark,
        )

        ledgers.append(iv_ledger)

    combined_ledger = combine_diagnostic_ledgers(*ledgers)
    combined_summary = summarize_diagnostic_ledger(combined_ledger)

    return {
        "surface": surface,
        "surface_with_iv": iv_surface,
        "ledger": combined_ledger,
        "summary": combined_summary,
        "pointwise_failures": pointwise_failures,
        "parity_failures": parity_failures,
        "vertical_failures": vertical_failures,
        "vertical_pairs": vertical_pairs,
        "convexity_failures": convexity_failures,
        "convexity_triplets": convexity_triplets,
        "convexity_slopes": convexity_slopes,
        "term_structure_failures": term_structure_failures,
        "term_structure_pairs": term_structure_pairs,
        "iv_failures": iv_failures,
        "iv_identifiability_summary": iv_identifiability_summary,
    }


# ------------------------------------------------------------
# Build negative-control registry
# ------------------------------------------------------------

negative_control_surfaces = {
    "call_below_lower_bound": corrupt_call_below_lower_bound(clean_surface),
    "put_above_upper_bound": corrupt_put_above_upper_bound(clean_surface),
    "call_vertical_violation": corrupt_call_vertical_monotonicity(clean_surface),
    "put_vertical_violation": corrupt_put_vertical_monotonicity(clean_surface),
    "call_butterfly_violation": corrupt_call_butterfly_convexity(clean_surface),
    "put_butterfly_violation": corrupt_put_butterfly_convexity(clean_surface),
    "put_call_parity_violation": corrupt_put_call_parity_only(clean_surface),
    "total_variance_calendar_violation": corrupt_total_variance_term_structure(clean_surface),
}


negative_control_registry_audit = pd.DataFrame(
    [
        {
            "negative_control": name,
            "surface_id": str(surface["surface_id"].iloc[0]),
            "rows": len(surface),
            "maturities": surface["T"].nunique(),
            "log_moneyness_points": surface["k_log"].nunique(),
            "status": STATUS_PASS,
        }
        for name, surface in negative_control_surfaces.items()
    ]
)

negative_control_registry_audit

,negative_control,surface_id,rows,maturities,log_moneyness_points,status
0,call_below_lower_bound,negative_call_below_lower_bound,55,5,11,PASS
1,put_above_upper_bound,negative_put_above_upper_bound,55,5,11,PASS
2,call_vertical_violation,negative_call_vertical_violation,55,5,11,PASS
3,put_vertical_violation,negative_put_vertical_violation,55,5,11,PASS
4,call_butterfly_violation,negative_call_butterfly_violation,55,5,11,PASS
5,put_butterfly_violation,negative_put_butterfly_violation,55,5,11,PASS
6,put_call_parity_violation,negative_put_call_parity_violation,55,5,11,PASS
7,total_variance_calendar_violation,negative_total_variance_calendar_violation,55,5,11,PASS


In [14]:
# ============================================================
# Cell 16: Run negative-control diagnostics and verify intended failures
# ============================================================

EXPECTED_NEGATIVE_CONTROL_FAILURES = {
    "call_below_lower_bound": [
        ("pointwise_bounds", "call_lower_bound"),
    ],
    "put_above_upper_bound": [
        ("pointwise_bounds", "put_upper_bound"),
    ],
    "call_vertical_violation": [
        ("vertical_spread_monotonicity", "call_nonincreasing_in_strike"),
    ],
    "put_vertical_violation": [
        ("vertical_spread_monotonicity", "put_nondecreasing_in_strike"),
    ],
    "call_butterfly_violation": [
        ("butterfly_convexity", "call_convex_in_strike"),
    ],
    "put_butterfly_violation": [
        ("butterfly_convexity", "put_convex_in_strike"),
    ],
    "put_call_parity_violation": [
        ("put_call_parity", "put_call_parity_residual"),
    ],
    "total_variance_calendar_violation": [
        ("calendar_total_variance", "total_variance_nondecreasing_in_maturity"),
    ],
}


def run_negative_control_suite(
    negative_surfaces: Dict[str, pd.DataFrame],
    *,
    run_iv_diagnostics: bool = True,
) -> Dict[str, Dict[str, object]]:
    """
    Run the full diagnostic suite on every negative-control surface.

    The result dictionary is keyed by negative-control name.
    """
    results = {}

    for control_name, surface in negative_surfaces.items():
        results[control_name] = run_static_arbitrage_diagnostic_suite(
            surface,
            clean_benchmark=False,
            run_iv_diagnostics=run_iv_diagnostics,
        )

    return results


def build_negative_control_expectation_report(
    negative_results: Dict[str, Dict[str, object]],
    expected_failures: Dict[str, List[Tuple[str, str]]],
) -> pd.DataFrame:
    """
    Verify that every negative control triggers its intended diagnostic failure.

    This report does not require exclusivity. A corrupted surface may trigger
    collateral failures in other diagnostics, especially because price-level
    corruption can also break put-call parity or IV inversion.

    The key question is:

        Did the targeted diagnostic fail?
    """
    rows = []

    for control_name, expected_pairs in expected_failures.items():
        if control_name not in negative_results:
            rows.append(
                {
                    "negative_control": control_name,
                    "expected_diagnostic_group": np.nan,
                    "expected_check_name": np.nan,
                    "observed_status": "MISSING_CONTROL",
                    "observed_failure_count": np.nan,
                    "observed_max_violation": np.nan,
                    "target_failure_detected": False,
                }
            )
            continue

        ledger = negative_results[control_name]["ledger"].copy()

        for diagnostic_group, check_name in expected_pairs:
            mask = (
                (ledger["diagnostic_group"] == diagnostic_group)
                & (ledger["check_name"] == check_name)
            )

            match = ledger.loc[mask]

            if len(match) != 1:
                rows.append(
                    {
                        "negative_control": control_name,
                        "expected_diagnostic_group": diagnostic_group,
                        "expected_check_name": check_name,
                        "observed_status": "MISSING_CHECK",
                        "observed_failure_count": np.nan,
                        "observed_max_violation": np.nan,
                        "target_failure_detected": False,
                    }
                )
                continue

            record = match.iloc[0]

            target_failure_detected = (
                str(record["status"]) == STATUS_FAIL
                and int(record["failure_count"]) > 0
                and float(record["max_violation"]) > float(record["tolerance"])
            )

            rows.append(
                {
                    "negative_control": control_name,
                    "expected_diagnostic_group": diagnostic_group,
                    "expected_check_name": check_name,
                    "observed_status": str(record["status"]),
                    "observed_failure_count": int(record["failure_count"]),
                    "observed_max_violation": float(record["max_violation"]),
                    "observed_tolerance": float(record["tolerance"]),
                    "observed_severity": str(record["severity"]),
                    "target_failure_detected": bool(target_failure_detected),
                }
            )

    return pd.DataFrame(rows)


def build_negative_control_failure_overview(
    negative_results: Dict[str, Dict[str, object]]
) -> pd.DataFrame:
    """
    Summarize total diagnostic failures by negative-control surface.
    """
    rows = []

    for control_name, result in negative_results.items():
        ledger = result["ledger"].copy()

        failed = ledger.loc[ledger["status"] == STATUS_FAIL].copy()

        rows.append(
            {
                "negative_control": control_name,
                "surface_id": str(ledger["surface_id"].iloc[0]) if len(ledger) > 0 else "",
                "total_checks": int(len(ledger)),
                "failed_checks": int(len(failed)),
                "total_failure_count": int(ledger["failure_count"].sum()),
                "max_violation": float(ledger["max_violation"].max()) if len(ledger) > 0 else np.nan,
                "failed_check_names": ", ".join(failed["check_name"].astype(str).tolist()),
            }
        )

    return pd.DataFrame(rows)


def combine_negative_control_ledgers(
    negative_results: Dict[str, Dict[str, object]]
) -> pd.DataFrame:
    """
    Combine all negative-control ledgers into one table with a control-name column.
    """
    ledgers = []

    for control_name, result in negative_results.items():
        ledger = result["ledger"].copy()
        ledger.insert(0, "negative_control", control_name)
        ledgers.append(ledger)

    if len(ledgers) == 0:
        return pd.DataFrame()

    return pd.concat(ledgers, ignore_index=True)


# ------------------------------------------------------------
# Run all negative controls
# ------------------------------------------------------------

negative_control_results = run_negative_control_suite(
    negative_control_surfaces,
    run_iv_diagnostics=True,
)

negative_control_expectation_report = build_negative_control_expectation_report(
    negative_control_results,
    EXPECTED_NEGATIVE_CONTROL_FAILURES,
)

negative_control_failure_overview = build_negative_control_failure_overview(
    negative_control_results
)

negative_control_ledger = combine_negative_control_ledgers(
    negative_control_results
)


# ------------------------------------------------------------
# Hard assertion: every targeted negative-control failure must be detected
# ------------------------------------------------------------

if not bool(negative_control_expectation_report["target_failure_detected"].all()):
    display(negative_control_expectation_report)
    display(negative_control_failure_overview)
    raise AssertionError(
        "At least one negative control failed to trigger its intended diagnostic."
    )


negative_control_expectation_report

,negative_control,expected_diagnostic_group,expected_check_name,observed_status,observed_failure_count,observed_max_violation,observed_tolerance,observed_severity,target_failure_detected
0,call_below_lower_bound,pointwise_bounds,call_lower_bound,FAIL,1,0.0500000000,0.0000000001,HIGH,True
1,put_above_upper_bound,pointwise_bounds,put_upper_bound,FAIL,1,0.0500000000,0.0000000001,HIGH,True
2,call_vertical_violation,vertical_spread_monotonicity,call_nonincreasing_in_strike,FAIL,1,0.1000000000,0.0000000001,HIGH,True
3,put_vertical_violation,vertical_spread_monotonicity,put_nondecreasing_in_strike,FAIL,1,0.1000000000,0.0000000001,HIGH,True
4,call_butterfly_violation,butterfly_convexity,call_convex_in_strike,FAIL,1,1.0651088866,0.0000000001,HIGH,True
5,put_butterfly_violation,butterfly_convexity,put_convex_in_strike,FAIL,1,1.2731431018,0.0000000001,HIGH,True
6,put_call_parity_violation,put_call_parity,put_call_parity_residual,FAIL,1,0.2500000000,0.0000000010,HIGH,True
7,total_variance_calendar_violation,calendar_total_variance,total_variance_nondecreasing_in_maturity,FAIL,1,0.0150000000,0.0000000001,HIGH,True


In [15]:
# ============================================================
# Cell 17: Negative-control failure matrix and collateral diagnostics
# ============================================================

STRUCTURAL_DIAGNOSTIC_ORDER = [
    ("pointwise_bounds", "call_lower_bound"),
    ("pointwise_bounds", "call_upper_bound"),
    ("pointwise_bounds", "put_lower_bound"),
    ("pointwise_bounds", "put_upper_bound"),
    ("put_call_parity", "put_call_parity_residual"),
    ("vertical_spread_monotonicity", "call_nonincreasing_in_strike"),
    ("vertical_spread_monotonicity", "put_nondecreasing_in_strike"),
    ("butterfly_convexity", "call_convex_in_strike"),
    ("butterfly_convexity", "put_convex_in_strike"),
    ("calendar_total_variance", "total_variance_nondecreasing_in_maturity"),
]


def diagnostic_key(diagnostic_group: str, check_name: str) -> str:
    """
    Compact diagnostic key for matrix-style summaries.
    """
    return f"{diagnostic_group}::{check_name}"


STRUCTURAL_DIAGNOSTIC_KEYS = [
    diagnostic_key(group, check)
    for group, check in STRUCTURAL_DIAGNOSTIC_ORDER
]


def add_diagnostic_key_column(ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Add a compact diagnostic_key column to a diagnostic ledger.
    """
    require_columns(
        ledger,
        ["diagnostic_group", "check_name"],
        table_name="diagnostic ledger",
    )

    out = ledger.copy()
    out["diagnostic_key"] = [
        diagnostic_key(group, check)
        for group, check in zip(out["diagnostic_group"], out["check_name"])
    ]

    return out


def build_structural_status_matrix(negative_ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Build a PASS / FAIL matrix for structural no-arbitrage diagnostics.

    Rows:
        negative-control surfaces

    Columns:
        structural diagnostic checks

    Values:
        PASS / FAIL
    """
    require_columns(
        negative_ledger,
        [
            "negative_control",
            "diagnostic_group",
            "check_name",
            "status",
        ],
        table_name="negative-control ledger",
    )

    work = add_diagnostic_key_column(negative_ledger)

    work = work.loc[
        work["diagnostic_key"].isin(STRUCTURAL_DIAGNOSTIC_KEYS)
    ].copy()

    matrix = work.pivot_table(
        index="negative_control",
        columns="diagnostic_key",
        values="status",
        aggfunc="first",
    )

    matrix = matrix.reindex(
        index=list(negative_control_surfaces.keys()),
        columns=STRUCTURAL_DIAGNOSTIC_KEYS,
    )

    return matrix


def build_structural_failure_count_matrix(negative_ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Build a structural diagnostic failure-count matrix.
    """
    require_columns(
        negative_ledger,
        [
            "negative_control",
            "diagnostic_group",
            "check_name",
            "failure_count",
        ],
        table_name="negative-control ledger",
    )

    work = add_diagnostic_key_column(negative_ledger)

    work = work.loc[
        work["diagnostic_key"].isin(STRUCTURAL_DIAGNOSTIC_KEYS)
    ].copy()

    matrix = work.pivot_table(
        index="negative_control",
        columns="diagnostic_key",
        values="failure_count",
        aggfunc="sum",
        fill_value=0,
    )

    matrix = matrix.reindex(
        index=list(negative_control_surfaces.keys()),
        columns=STRUCTURAL_DIAGNOSTIC_KEYS,
        fill_value=0,
    )

    return matrix.astype(int)


def build_structural_max_violation_matrix(negative_ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Build a structural diagnostic maximum-violation matrix.
    """
    require_columns(
        negative_ledger,
        [
            "negative_control",
            "diagnostic_group",
            "check_name",
            "max_violation",
        ],
        table_name="negative-control ledger",
    )

    work = add_diagnostic_key_column(negative_ledger)

    work = work.loc[
        work["diagnostic_key"].isin(STRUCTURAL_DIAGNOSTIC_KEYS)
    ].copy()

    matrix = work.pivot_table(
        index="negative_control",
        columns="diagnostic_key",
        values="max_violation",
        aggfunc="max",
        fill_value=0.0,
    )

    matrix = matrix.reindex(
        index=list(negative_control_surfaces.keys()),
        columns=STRUCTURAL_DIAGNOSTIC_KEYS,
        fill_value=0.0,
    )

    return matrix.astype(float)


def build_target_vs_collateral_report(
    negative_ledger: pd.DataFrame,
    expected_failures: Dict[str, List[Tuple[str, str]]],
) -> pd.DataFrame:
    """
    Separate intended target failures from collateral failures.

    A collateral failure is not automatically bad. Some corruptions naturally
    break multiple no-arbitrage relations. This report simply makes the
    dependence structure explicit.
    """
    require_columns(
        negative_ledger,
        [
            "negative_control",
            "diagnostic_group",
            "check_name",
            "status",
            "failure_count",
            "max_violation",
        ],
        table_name="negative-control ledger",
    )

    work = add_diagnostic_key_column(negative_ledger)

    structural = work.loc[
        work["diagnostic_key"].isin(STRUCTURAL_DIAGNOSTIC_KEYS)
    ].copy()

    rows = []

    for control_name, group in structural.groupby("negative_control", sort=False):
        expected_keys = {
            diagnostic_key(group_name, check_name)
            for group_name, check_name in expected_failures.get(control_name, [])
        }

        failed = group.loc[group["status"] == STATUS_FAIL].copy()

        target_failed = failed.loc[
            failed["diagnostic_key"].isin(expected_keys)
        ].copy()

        collateral_failed = failed.loc[
            ~failed["diagnostic_key"].isin(expected_keys)
        ].copy()

        rows.append(
            {
                "negative_control": control_name,
                "target_failure_count": int(len(target_failed)),
                "collateral_failure_count": int(len(collateral_failed)),
                "total_structural_failed_checks": int(len(failed)),
                "target_failed_checks": ", ".join(target_failed["diagnostic_key"].astype(str).tolist()),
                "collateral_failed_checks": ", ".join(collateral_failed["diagnostic_key"].astype(str).tolist()),
                "max_structural_violation": float(failed["max_violation"].max()) if len(failed) > 0 else 0.0,
            }
        )

    report = pd.DataFrame(rows)

    report = report.set_index("negative_control").reindex(
        list(negative_control_surfaces.keys())
    ).reset_index()

    return report


# ------------------------------------------------------------
# Build negative-control matrices
# ------------------------------------------------------------

negative_structural_status_matrix = build_structural_status_matrix(
    negative_control_ledger
)

negative_structural_failure_count_matrix = build_structural_failure_count_matrix(
    negative_control_ledger
)

negative_structural_max_violation_matrix = build_structural_max_violation_matrix(
    negative_control_ledger
)

negative_target_vs_collateral_report = build_target_vs_collateral_report(
    negative_control_ledger,
    EXPECTED_NEGATIVE_CONTROL_FAILURES,
)


# ------------------------------------------------------------
# Compact display matrix
# ------------------------------------------------------------

negative_structural_display_matrix = negative_structural_status_matrix.copy()

negative_structural_display_matrix = negative_structural_display_matrix.replace(
    {
        STATUS_PASS: ".",
        STATUS_FAIL: "FAIL",
    }
)

negative_structural_display_matrix

diagnostic_key,pointwise_bounds::call_lower_bound,pointwise_bounds::call_upper_bound,pointwise_bounds::put_lower_bound,pointwise_bounds::put_upper_bound,put_call_parity::put_call_parity_residual,vertical_spread_monotonicity::call_nonincreasing_in_strike,vertical_spread_monotonicity::put_nondecreasing_in_strike,butterfly_convexity::call_convex_in_strike,butterfly_convexity::put_convex_in_strike,calendar_total_variance::total_variance_nondecreasing_in_maturity
negative_control,,,,,,,,,,
call_below_lower_bound,FAIL,.,.,.,FAIL,.,.,FAIL,.,.
put_above_upper_bound,.,.,.,FAIL,FAIL,.,FAIL,.,FAIL,.
call_vertical_violation,.,.,.,.,FAIL,FAIL,.,FAIL,.,.
put_vertical_violation,.,.,.,.,FAIL,.,FAIL,.,FAIL,.
call_butterfly_violation,.,.,.,.,FAIL,FAIL,.,FAIL,.,.
put_butterfly_violation,.,.,.,.,FAIL,.,FAIL,.,FAIL,.
put_call_parity_violation,.,.,.,.,FAIL,.,.,.,.,.
total_variance_calendar_violation,.,.,.,.,.,.,.,.,.,FAIL


In [16]:
# ============================================================
# Cell 18: Cross-diagnostic interpretation of negative controls
# ============================================================

def build_negative_control_interpretation_report(
    expectation_report: pd.DataFrame,
    target_vs_collateral_report: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build an interpretation report for negative-control diagnostics.

    The purpose is to distinguish:

    1. target detection:
       The intended diagnostic failed.

    2. collateral detection:
       Other diagnostics also failed because the corruption changed prices in a
       way that mechanically broke additional no-arbitrage relationships.

    Collateral failures are not automatically implementation errors. They often
    confirm that static-arbitrage conditions are interdependent.
    """
    require_columns(
        expectation_report,
        [
            "negative_control",
            "expected_diagnostic_group",
            "expected_check_name",
            "observed_status",
            "observed_failure_count",
            "observed_max_violation",
            "target_failure_detected",
        ],
        table_name="negative-control expectation report",
    )

    require_columns(
        target_vs_collateral_report,
        [
            "negative_control",
            "target_failure_count",
            "collateral_failure_count",
            "total_structural_failed_checks",
            "target_failed_checks",
            "collateral_failed_checks",
            "max_structural_violation",
        ],
        table_name="target-vs-collateral report",
    )

    merged = expectation_report.merge(
        target_vs_collateral_report,
        on="negative_control",
        how="left",
    )

    interpretation_rows = []

    for row in merged.itertuples(index=False):
        target_detected = bool(row.target_failure_detected)
        collateral_count = int(row.collateral_failure_count)

        if not target_detected:
            interpretation = "FAILED: intended diagnostic was not triggered."
            status = STATUS_FAIL
        elif collateral_count == 0:
            interpretation = "Target diagnostic triggered cleanly with no collateral structural failures."
            status = STATUS_PASS
        else:
            interpretation = (
                "Target diagnostic triggered; collateral failures are present because the injected "
                "price corruption also breaks related static-arbitrage relations."
            )
            status = STATUS_PASS

        interpretation_rows.append(
            {
                "negative_control": row.negative_control,
                "expected_target": diagnostic_key(
                    row.expected_diagnostic_group,
                    row.expected_check_name,
                ),
                "target_failure_detected": target_detected,
                "target_failure_count": int(row.target_failure_count),
                "collateral_failure_count": collateral_count,
                "total_structural_failed_checks": int(row.total_structural_failed_checks),
                "observed_max_target_violation": float(row.observed_max_violation),
                "max_structural_violation": float(row.max_structural_violation),
                "target_failed_checks": row.target_failed_checks,
                "collateral_failed_checks": row.collateral_failed_checks,
                "interpretation_status": status,
                "interpretation": interpretation,
            }
        )

    return pd.DataFrame(interpretation_rows)


def build_diagnostic_sensitivity_report(
    negative_structural_failure_counts: pd.DataFrame,
) -> pd.DataFrame:
    """
    Count how often each structural diagnostic fires across negative controls.

    This helps identify diagnostics that are highly local versus diagnostics
    that often fire as collateral checks.
    """
    if negative_structural_failure_counts is None or len(negative_structural_failure_counts) == 0:
        return pd.DataFrame()

    failure_binary = (negative_structural_failure_counts > 0).astype(int)

    rows = []

    for diagnostic_col in failure_binary.columns:
        rows.append(
            {
                "diagnostic_key": diagnostic_col,
                "negative_controls_triggered": int(failure_binary[diagnostic_col].sum()),
                "total_failure_rows": int(negative_structural_failure_counts[diagnostic_col].sum()),
                "trigger_rate": float(failure_binary[diagnostic_col].mean()),
            }
        )

    out = pd.DataFrame(rows)

    out = out.sort_values(
        ["negative_controls_triggered", "total_failure_rows", "diagnostic_key"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    return out


negative_control_interpretation_report = build_negative_control_interpretation_report(
    negative_control_expectation_report,
    negative_target_vs_collateral_report,
)

negative_diagnostic_sensitivity_report = build_diagnostic_sensitivity_report(
    negative_structural_failure_count_matrix,
)


# ------------------------------------------------------------
# Hard assertion: interpretation layer must preserve target detection
# ------------------------------------------------------------

if not bool((negative_control_interpretation_report["interpretation_status"] == STATUS_PASS).all()):
    display(negative_control_interpretation_report)
    raise AssertionError(
        "Negative-control interpretation failed because at least one target diagnostic was not detected."
    )


negative_control_interpretation_report

,negative_control,expected_target,target_failure_detected,target_failure_count,collateral_failure_count,total_structural_failed_checks,observed_max_target_violation,max_structural_violation,target_failed_checks,collateral_failed_checks,interpretation_status,interpretation
0,call_below_lower_bound,pointwise_bounds::call_lower_bound,True,1,2,3,0.0500000000,0.1265599443,pointwise_bounds::call_lower_bound,"put_call_parity::put_call_parity_residual, but...",PASS,Target diagnostic triggered; collateral failur...
1,put_above_upper_bound,pointwise_bounds::put_upper_bound,True,1,3,4,0.0500000000,99.7068017126,pointwise_bounds::put_upper_bound,"put_call_parity::put_call_parity_residual, ver...",PASS,Target diagnostic triggered; collateral failur...
2,call_vertical_violation,vertical_spread_monotonicity::call_nonincreasi...,True,1,2,3,0.1000000000,2.1518066496,vertical_spread_monotonicity::call_nonincreasi...,"put_call_parity::put_call_parity_residual, but...",PASS,Target diagnostic triggered; collateral failur...
3,put_vertical_violation,vertical_spread_monotonicity::put_nondecreasin...,True,1,2,3,0.1000000000,2.3252970224,vertical_spread_monotonicity::put_nondecreasin...,"put_call_parity::put_call_parity_residual, but...",PASS,Target diagnostic triggered; collateral failur...
4,call_butterfly_violation,butterfly_convexity::call_convex_in_strike,True,1,2,3,1.0651088866,2.9602897933,butterfly_convexity::call_convex_in_strike,"put_call_parity::put_call_parity_residual, ver...",PASS,Target diagnostic triggered; collateral failur...
5,put_butterfly_violation,butterfly_convexity::put_convex_in_strike,True,1,2,3,1.2731431018,3.4907712436,butterfly_convexity::put_convex_in_strike,"put_call_parity::put_call_parity_residual, ver...",PASS,Target diagnostic triggered; collateral failur...
6,put_call_parity_violation,put_call_parity::put_call_parity_residual,True,1,0,1,0.2500000000,0.2500000000,put_call_parity::put_call_parity_residual,,PASS,Target diagnostic triggered cleanly with no co...
7,total_variance_calendar_violation,calendar_total_variance::total_variance_nondec...,True,1,0,1,0.0150000000,0.0150000000,calendar_total_variance::total_variance_nondec...,,PASS,Target diagnostic triggered cleanly with no co...


In [17]:
# ============================================================
# Cell 19: Final validation ledger and notebook readiness gate
# ============================================================

def _metric_value(report: pd.DataFrame, metric_name: str):
    """
    Extract one metric value from a two-column report with columns:

        metric, value
    """
    require_columns(
        report,
        ["metric", "value"],
        table_name="metric report",
    )

    match = report.loc[report["metric"] == metric_name, "value"]

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one metric named {metric_name}; found {len(match)}."
        )

    return match.iloc[0]


def make_validation_record(
    *,
    validation_check: str,
    passed: bool,
    observed_value,
    required_value,
    message: str,
) -> Dict[str, object]:
    """
    Build one final validation ledger record.
    """
    return {
        "validation_check": validation_check,
        "status": STATUS_PASS if bool(passed) else STATUS_FAIL,
        "observed_value": observed_value,
        "required_value": required_value,
        "message": message,
    }


def build_final_notebook_validation_ledger() -> pd.DataFrame:
    """
    Build the final notebook validation ledger.

    Notebook 07 is considered ready only if:

    1. the clean Black-Scholes positive-control surface passes all diagnostics;
    2. all clean failure-detail tables are empty;
    3. all required structural diagnostic groups are present;
    4. every negative control triggers its intended diagnostic;
    5. every negative-control interpretation passes;
    6. the structural failure matrix has no missing entries;
    7. every negative-control surface has at least one structural failure;
    8. the expected number of negative controls was tested.
    """
    required_clean_groups = {
        "pointwise_bounds",
        "put_call_parity",
        "vertical_spread_monotonicity",
        "butterfly_convexity",
        "calendar_total_variance",
        "implied_volatility_consistency",
    }

    observed_clean_groups = set(clean_positive_control_ledger["diagnostic_group"].astype(str))

    clean_ready = bool(
        _metric_value(
            clean_positive_control_report,
            "clean_positive_control_ready",
        )
    )

    clean_failed_checks = int(
        _metric_value(
            clean_positive_control_report,
            "failed_diagnostic_checks",
        )
    )

    clean_total_failures = int(
        _metric_value(
            clean_positive_control_report,
            "total_failure_count",
        )
    )

    clean_failure_tables_empty = bool(
        (clean_failure_detail_counts["failure_rows"].astype(int) == 0).all()
    )

    clean_groups_present = bool(
        required_clean_groups.issubset(observed_clean_groups)
    )

    negative_target_detection = bool(
        negative_control_expectation_report["target_failure_detected"].astype(bool).all()
    )

    negative_interpretation_pass = bool(
        (negative_control_interpretation_report["interpretation_status"] == STATUS_PASS).all()
    )

    expected_negative_count = len(EXPECTED_NEGATIVE_CONTROL_FAILURES)
    observed_negative_count = int(negative_control_expectation_report["negative_control"].nunique())

    negative_count_correct = bool(
        observed_negative_count == expected_negative_count
    )

    structural_status_matrix_complete = bool(
        not negative_structural_status_matrix.isna().any().any()
    )

    every_negative_has_structural_failure = bool(
        (negative_structural_failure_count_matrix.sum(axis=1).astype(int) > 0).all()
    )

    expected_target_pairs_tested = int(
        sum(len(v) for v in EXPECTED_NEGATIVE_CONTROL_FAILURES.values())
    )

    observed_target_pairs_tested = int(len(negative_control_expectation_report))

    target_pair_count_correct = bool(
        observed_target_pairs_tested == expected_target_pairs_tested
    )

    records = [
        make_validation_record(
            validation_check="clean_positive_control_ready",
            passed=clean_ready,
            observed_value=clean_ready,
            required_value=True,
            message="Clean Black-Scholes benchmark surface must pass every diagnostic.",
        ),
        make_validation_record(
            validation_check="clean_failed_checks_zero",
            passed=clean_failed_checks == 0,
            observed_value=clean_failed_checks,
            required_value=0,
            message="Clean positive control must have zero failed diagnostic checks.",
        ),
        make_validation_record(
            validation_check="clean_total_failure_count_zero",
            passed=clean_total_failures == 0,
            observed_value=clean_total_failures,
            required_value=0,
            message="Clean positive control must have zero row-level diagnostic failures.",
        ),
        make_validation_record(
            validation_check="clean_failure_detail_tables_empty",
            passed=clean_failure_tables_empty,
            observed_value=int(clean_failure_detail_counts["failure_rows"].sum()),
            required_value=0,
            message="All clean positive-control failure-detail tables must be empty.",
        ),
        make_validation_record(
            validation_check="required_clean_diagnostic_groups_present",
            passed=clean_groups_present,
            observed_value=", ".join(sorted(observed_clean_groups)),
            required_value=", ".join(sorted(required_clean_groups)),
            message="Clean ledger must include all required diagnostic groups.",
        ),
        make_validation_record(
            validation_check="negative_control_count_correct",
            passed=negative_count_correct,
            observed_value=observed_negative_count,
            required_value=expected_negative_count,
            message="All planned negative-control surfaces must be tested.",
        ),
        make_validation_record(
            validation_check="negative_target_pair_count_correct",
            passed=target_pair_count_correct,
            observed_value=observed_target_pairs_tested,
            required_value=expected_target_pairs_tested,
            message="All expected negative-control target failures must be represented.",
        ),
        make_validation_record(
            validation_check="negative_targets_detected",
            passed=negative_target_detection,
            observed_value=int(negative_control_expectation_report["target_failure_detected"].sum()),
            required_value=expected_target_pairs_tested,
            message="Every negative control must trigger its intended diagnostic failure.",
        ),
        make_validation_record(
            validation_check="negative_interpretations_pass",
            passed=negative_interpretation_pass,
            observed_value=int((negative_control_interpretation_report["interpretation_status"] == STATUS_PASS).sum()),
            required_value=len(negative_control_interpretation_report),
            message="Every negative-control interpretation must preserve target detection.",
        ),
        make_validation_record(
            validation_check="structural_status_matrix_complete",
            passed=structural_status_matrix_complete,
            observed_value=int(negative_structural_status_matrix.isna().sum().sum()),
            required_value=0,
            message="Structural diagnostic status matrix must not contain missing entries.",
        ),
        make_validation_record(
            validation_check="every_negative_has_structural_failure",
            passed=every_negative_has_structural_failure,
            observed_value=int((negative_structural_failure_count_matrix.sum(axis=1).astype(int) > 0).sum()),
            required_value=expected_negative_count,
            message="Every negative-control surface must produce at least one structural diagnostic failure.",
        ),
    ]

    validation_ledger = pd.DataFrame(records)

    return validation_ledger


final_validation_ledger = build_final_notebook_validation_ledger()

NOTEBOOK_07_READY = bool((final_validation_ledger["status"] == STATUS_PASS).all())

if not NOTEBOOK_07_READY:
    display(final_validation_ledger)
    raise AssertionError(
        "NOTEBOOK_07_READY is False because at least one final validation check failed."
    )


final_validation_ledger

,validation_check,status,observed_value,required_value,message
0,clean_positive_control_ready,PASS,True,True,Clean Black-Scholes benchmark surface must pas...
1,clean_failed_checks_zero,PASS,0,0,Clean positive control must have zero failed d...
2,clean_total_failure_count_zero,PASS,0,0,Clean positive control must have zero row-leve...
3,clean_failure_detail_tables_empty,PASS,0,0,All clean positive-control failure-detail tabl...
4,required_clean_diagnostic_groups_present,PASS,"butterfly_convexity, calendar_total_variance, ...","butterfly_convexity, calendar_total_variance, ...",Clean ledger must include all required diagnos...
5,negative_control_count_correct,PASS,8,8,All planned negative-control surfaces must be ...
6,negative_target_pair_count_correct,PASS,8,8,All expected negative-control target failures ...
7,negative_targets_detected,PASS,8,8,Every negative control must trigger its intend...
8,negative_interpretations_pass,PASS,8,8,Every negative-control interpretation must pre...
9,structural_status_matrix_complete,PASS,0,0,Structural diagnostic status matrix must not c...


## Final research conclusion

This notebook built and validated a static no-arbitrage diagnostic layer for synthetic European option surfaces.

The diagnostic framework was tested in two stages.

First, a clean Black-Scholes benchmark surface was used as the positive control. The clean surface passed all implemented diagnostics:

- pointwise call and put bounds;
- put-call parity;
- vertical-spread monotonicity;
- butterfly convexity;
- total-variance term-structure consistency;
- implied-volatility inversion and repricing consistency.

The clean benchmark produced zero failed diagnostic checks and zero row-level failures.

Second, deliberately corrupted negative-control surfaces were constructed. Each negative control was designed to violate a specific no-arbitrage condition. The diagnostic framework correctly detected every intended failure:

- call below lower bound;
- put above upper bound;
- call vertical-spread violation;
- put vertical-spread violation;
- call butterfly-convexity violation;
- put butterfly-convexity violation;
- put-call parity violation;
- total-variance calendar violation.

Some corrupted price surfaces also triggered collateral failures in related diagnostics. This is expected. Static no-arbitrage conditions are interdependent. A price corruption that targets one condition can mechanically break parity, convexity, monotonicity, or implied-volatility consistency at the same time.

The final validation ledger passed every readiness check.

Therefore, this notebook supports the following claim:

> A controlled synthetic European option surface can be passed through a reusable static-arbitrage diagnostic framework that correctly accepts a clean Black-Scholes surface and correctly rejects deliberately corrupted surfaces.

The notebook does not claim that the framework repairs bad surfaces, validates real option-chain data, handles American options, incorporates bid-ask microstructure, or proves that a fitted volatility surface is globally arbitrage-free.

The diagnostic layer is now ready to be reused before empirical option-chain ingestion or volatility-surface fitting.

`NOTEBOOK_07_READY = True`